# ARC-AGI-3 Kaggle Competition — TRUE SCORED RUN

**Control target:** `wellkilo/arc3lab-duck-v12-control-seed-20260819`  
**Recorded control score:** `3.57`  
**Seed:** `20260819`  
**Required analyzer:** `Qwen/Qwen3.8-27B-FP8`

This is a true scored-run-only notebook. It declares the ARC competition + Duck/TAAF + offline vLLM inputs and attaches the exact Qwen3.8 Kaggle Model source. Every required component is resolved and validated before any ARC move is allowed:

1. ARC Prize 2026 — ARC-AGI-3 competition input
2. `jeroencottaar/taaf-kaggle-source-share`
3. `driessmit1/arc3-vllm-h100-wheelhouse-v3`
4. Kaggle Model `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1`

Runtime order:

`AUTOLOAD inputs → validate exact Qwen3.8 snapshot → install ARC offline wheels → publish one canonical TAAF input map (including legacy Qwen3.6 lookup alias redirected to Qwen3.8) → rewrite immutable setup model ID to Qwen3.8 → run official source setup → reconcile pinned vLLM runtime → verify GPU + imports + /v1/models + real completion → GhostBridge PRE-MOVE → ADL A/B → exactly one real action → POST_MOVE_ADL → next move → submission.parquet`

No internet downloads, no git clone, no recursive project dependency installation, no hidden second environment pass, no recovery probe actions, and no Qwen3.6 fallback. Missing inputs or a mismatched model/server fail closed before gameplay.


In [ ]:

import json
import os
import pickle
import random
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

CONTROL_SEED = int(os.environ.get("ADLDB_CONTROL_SEED", "20260819"))
KNOWN_PUBLIC_CONTROL_SEEDS = (20260819, 20260807)
ANALYZER_MODEL_ID = "Qwen/Qwen3.8-27B-FP8"
ANALYZER_CONTEXT_WINDOW = 32768

os.environ["PYTHONHASHSEED"] = str(CONTROL_SEED)
os.environ["ADLDB_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["TAAF_CONTROL_SEED"] = str(CONTROL_SEED)
os.environ["VLLM_SEED"] = str(CONTROL_SEED)

random.seed(CONTROL_SEED)
try:
    import numpy as _np
    _np.random.seed(CONTROL_SEED)
except Exception:
    _np = None
try:
    import torch as _torch
    _torch.manual_seed(CONTROL_SEED)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(CONTROL_SEED)
except Exception:
    _torch = None

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"

os.environ["INFERENCE_ANALYZER_MODEL"] = ANALYZER_MODEL_ID
os.environ["LOCAL_ANALYZER_MODEL_ID"] = ANALYZER_MODEL_ID
os.environ.setdefault("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
os.environ["TAAF_MAX_OUTPUT_TOKENS"] = "8192"
os.environ["TAAF_TOOL_STEPS"] = "8"
os.environ["TAAF_TEMPERATURE"] = "0.6"
os.environ["TAAF_TOP_P"] = "0.95"
os.environ["TAAF_CONTEXT_WINDOW"] = str(ANALYZER_CONTEXT_WINDOW)

# Stronger Duck-v12 perception request. If the mounted source bundle does not
# implement full-frame mode this flag is inert; the notebook's DWE/no-impact
# layer remains independent.
os.environ["ARC3_FRAME_MODE"] = "full"
os.environ["ARC3_STATE_GRAPH"] = "off"
os.environ["ARC3_REEXPLORE_STRICT"] = "0"
os.environ["ADLDB_NO_IMPACT"] = "on"

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry
    for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)]
    if entry
)

WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(
    "ADLDB QWEN38 CONTROL-SEED "
    f"seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} "
    f"context={ANALYZER_CONTEXT_WINDOW} "
    f"frame_mode={os.environ['ARC3_FRAME_MODE']} "
    f"TRUE_SUBMISSION={TRUE_SUBMISSION}",
    flush=True,
)


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).


In [ ]:
# === AUTOLOAD STAGE 1 — RESOLVE + VALIDATE EVERY SCORED-RUN INPUT ===
# Qwen3.8 is an attached Kaggle Model, not a dataset. Fail closed on any other
# model generation and publish ONE canonical input map for the immutable Duck/TAAF setup.
from pathlib import Path
import json
import os
import sys

REQUIRED_COMPETITION = "arc-prize-2026-arc-agi-3"
TAAF_SOURCE_REF = "jeroencottaar/taaf-kaggle-source-share"
VLLM_WHEELHOUSE_REF = "driessmit1/arc3-vllm-h100-wheelhouse-v3"
QWEN38_MODEL_SOURCE = "foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
QWEN38_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN38_EXPECTED_MOUNT = Path(
    "/kaggle/input/models/foysalemonshanto/"
    "qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
)
LEGACY_TAAF_MODEL_REF = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"
REQUIRED_DATASET_SOURCES = [TAAF_SOURCE_REF, VLLM_WHEELHOUSE_REF]
DATASET_SOURCES = list(REQUIRED_DATASET_SOURCES)
MODEL_SOURCES = [QWEN38_MODEL_SOURCE]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
AUTO_INPUT_MANIFEST_PATH = WORKING_DIR / "auto_input_manifest.json"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_MODEL_ROOT = Path("/kaggle/models")


def _first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)


def _find_named(root: Path, name: str):
    if not root.exists():
        return None
    direct = root / name
    if direct.exists():
        return direct
    try:
        return next(root.rglob(name), None)
    except OSError:
        return None


def _require(path, label):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(
            f"AUTOLOAD REQUIRED INPUT MISSING: {label}. "
            "The notebook embeds competition/datasets/model sources; if Kaggle did not "
            "attach them, push with the included kernel-metadata-qwen38.json."
        )
    return Path(path).resolve()


def _dataset_candidates(ref: str):
    owner, slug = ref.split("/", 1)
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "datasets" / owner / slug,
    ]


def _competition_candidates(slug: str):
    return [
        KAGGLE_INPUT_ROOT / slug,
        KAGGLE_INPUT_ROOT / "competitions" / slug,
    ]


def _resolve_dataset(ref: str, marker: str | None = None):
    for candidate in _dataset_candidates(ref):
        if candidate.exists() and (marker is None or _find_named(candidate, marker) is not None):
            return candidate.resolve()
    if marker and KAGGLE_INPUT_ROOT.exists():
        try:
            hits = list(KAGGLE_INPUT_ROOT.rglob(marker))
        except OSError:
            hits = []
        roots = sorted({h.parent.resolve() for h in hits})
        if len(roots) == 1:
            return roots[0]
    raise FileNotFoundError(f"AUTOLOAD dataset not mounted: {ref}")


def _read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _has_model_weights(model_dir: Path):
    return (
        (model_dir / "model.safetensors").is_file()
        or (model_dir / "model.safetensors.index.json").is_file()
        or any(model_dir.glob("*.safetensors"))
    )


def _model_weight_files(model_dir: Path):
    index = model_dir / "model.safetensors.index.json"
    if index.is_file():
        payload = _read_json(index)
        names = sorted(set(str(x) for x in (payload.get("weight_map") or {}).values()))
        files = [model_dir / name for name in names]
        missing = [str(p) for p in files if not p.is_file()]
        if missing:
            raise FileNotFoundError("Qwen3.8 index references missing shards: " + ", ".join(missing[:20]))
        return files
    single = model_dir / "model.safetensors"
    if single.is_file():
        return [single]
    return sorted(model_dir.glob("*.safetensors"))


def _qwen38_identity_score(model_dir: Path, cfg: dict):
    text = (str(model_dir) + " " + json.dumps(cfg, sort_keys=True, default=str)).lower()
    score = 0
    for token, points in (
        ("qwen3.8", 240), ("qwen3-8", 230), ("qwen38", 220),
        ("27b", 50), ("fp8", 50), ("float8", 25), ("repacked", 20),
    ):
        if token in text:
            score += points
    if "qwen3.6" in text or "qwen3-6" in text or "qwen36" in text:
        score -= 2000
    qcfg = json.dumps(cfg.get("quantization_config", {}), sort_keys=True, default=str).lower()
    if "fp8" in qcfg or "float8" in qcfg:
        score += 25
    return score


def _candidate_qwen38_dirs():
    direct = [
        QWEN38_EXPECTED_MOUNT,
        Path("/kaggle/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"),
        Path("/kaggle/input/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"),
    ]
    yielded = set()
    for p in direct:
        if p.exists():
            rp = p.resolve()
            yielded.add(rp)
            yield rp
    # Controlled fallback for Kaggle mount-layout changes. Search only directories
    # whose path strongly identifies this exact Qwen3.8 model family.
    for root in (KAGGLE_INPUT_ROOT, KAGGLE_MODEL_ROOT):
        if not root.exists():
            continue
        try:
            cfgs = root.rglob("config.json")
        except OSError:
            continue
        for cfg_path in cfgs:
            model_dir = cfg_path.parent.resolve()
            low = str(model_dir).lower()
            if not (
                QWEN38_MODEL_SLUG in low
                or ("qwen3-8" in low and "27b" in low and "fp8" in low)
                or ("qwen3.8" in low and "27b" in low and "fp8" in low)
            ):
                continue
            if model_dir not in yielded:
                yielded.add(model_dir)
                yield model_dir


def _resolve_exact_qwen38():
    candidates = []
    for model_dir in _candidate_qwen38_dirs():
        cfg_path = model_dir / "config.json"
        if not cfg_path.is_file() or not _has_model_weights(model_dir):
            continue
        cfg = _read_json(cfg_path)
        candidates.append((_qwen38_identity_score(model_dir, cfg), model_dir, cfg))
    if not candidates:
        raise FileNotFoundError(
            "AUTOLOAD Qwen3.8 model source is attached but no complete HF snapshot was found. "
            f"Expected model source={QWEN38_MODEL_SOURCE} expected_mount={QWEN38_EXPECTED_MOUNT}"
        )
    candidates.sort(key=lambda x: (x[0], -len(str(x[1]))), reverse=True)
    score, model_dir, cfg = candidates[0]
    if score < 250:
        raise RuntimeError(
            f"AUTOLOAD WRONG MODEL: best candidate does not validate as Qwen3.8-27B-FP8: "
            f"{model_dir} identity_score={score}"
        )
    identity = (str(model_dir) + " " + json.dumps(cfg, sort_keys=True, default=str)).lower()
    if "qwen3.6" in identity or "qwen3-6" in identity or "qwen36" in identity:
        raise RuntimeError(f"AUTOLOAD WRONG MODEL GENERATION: Qwen3.6 resolved at {model_dir}")
    tokenizer_ok = (model_dir / "tokenizer_config.json").is_file() and (
        (model_dir / "tokenizer.json").is_file()
        or (model_dir / "vocab.json").is_file()
        or (model_dir / "tokenizer.model").is_file()
    )
    if not tokenizer_ok:
        raise FileNotFoundError(f"Qwen3.8 tokenizer payload incomplete under {model_dir}")
    weights = _model_weight_files(model_dir)
    if not weights:
        raise FileNotFoundError(f"No Qwen3.8 safetensors weights under {model_dir}")
    zero = [str(p) for p in weights if p.stat().st_size <= 0]
    if zero:
        raise RuntimeError("Zero-byte Qwen3.8 weight shards: " + ", ".join(zero[:20]))
    total_bytes = sum(p.stat().st_size for p in weights)
    if total_bytes < 10 * 1024**3:
        raise RuntimeError(f"Qwen3.8 payload unexpectedly small: {total_bytes / 1024**3:.2f} GiB")
    return model_dir, cfg, weights, total_bytes, score


# 1) Official competition mount.
ARC_COMPETITION_ROOT = _first_existing(_competition_candidates(REQUIRED_COMPETITION))
if ARC_COMPETITION_ROOT is None and KAGGLE_INPUT_ROOT.exists():
    hit = _find_named(KAGGLE_INPUT_ROOT, "arc_agi_3_wheels")
    if hit is not None and hit.is_dir():
        ARC_COMPETITION_ROOT = hit.parent
ARC_COMPETITION_ROOT = _require(ARC_COMPETITION_ROOT, f"competition:{REQUIRED_COMPETITION}")
ARC_WHEELS_DIR = ARC_COMPETITION_ROOT / "arc_agi_3_wheels"
if not ARC_WHEELS_DIR.is_dir():
    hit = _find_named(ARC_COMPETITION_ROOT, "arc_agi_3_wheels")
    ARC_WHEELS_DIR = _require(hit if hit is not None and hit.is_dir() else None, "arc_agi_3_wheels")
else:
    ARC_WHEELS_DIR = ARC_WHEELS_DIR.resolve()
ARC_ENVIRONMENTS_DIR = ARC_COMPETITION_ROOT / "environment_files"
if not TRUE_SUBMISSION:
    ARC_ENVIRONMENTS_DIR = _require(ARC_ENVIRONMENTS_DIR if ARC_ENVIRONMENTS_DIR.is_dir() else None, "offline environment_files")
else:
    ARC_ENVIRONMENTS_DIR = ARC_ENVIRONMENTS_DIR.resolve()

# 2) TAAF/Duck source bundle.
TAAF_BUNDLE_MOUNT = _resolve_dataset(TAAF_SOURCE_REF, DATASET_BUNDLE_MARKER)
bundle_marker = _find_named(TAAF_BUNDLE_MOUNT, DATASET_BUNDLE_MARKER)
BUNDLE_DIR = _require(bundle_marker.parent if bundle_marker else None, "TAAF source bundle marker")
for required_name in ("src", "setup_commands.json", "teardown_commands.json", "deploy_target.pkl", "benchmark_initial.pkl"):
    _require(BUNDLE_DIR / required_name, f"TAAF bundle component:{required_name}")

# 3) Offline vLLM wheelhouse and exact lock.
VLLM_WHEELHOUSE_MOUNT = _resolve_dataset(VLLM_WHEELHOUSE_REF, "requirements.lock")
wheel_lock = _find_named(VLLM_WHEELHOUSE_MOUNT, "requirements.lock")
VLLM_WHEELHOUSE_DIR = _require(wheel_lock.parent if wheel_lock else None, "vLLM wheelhouse requirements.lock")
REQUIREMENTS_LOCK = _require(VLLM_WHEELHOUSE_DIR / "requirements.lock", "vLLM requirements.lock")
wheel_files = list(VLLM_WHEELHOUSE_DIR.rglob("*.whl"))
if not wheel_files:
    raise FileNotFoundError(f"AUTOLOAD vLLM wheelhouse contains no .whl files: {VLLM_WHEELHOUSE_DIR}")

# 4) Exact Qwen3.8-27B-FP8 Kaggle Model snapshot.
QWEN_MODEL_DIR, QWEN_MODEL_CONFIG, QWEN_WEIGHT_FILES, QWEN_TOTAL_BYTES, QWEN_IDENTITY_SCORE = _resolve_exact_qwen38()

# One canonical input map. The legacy Qwen3.6 dataset key is retained ONLY as
# an immutable-TAAF lookup alias; its value points to the verified Qwen3.8 path.
kaggle_input_paths = {
    TAAF_SOURCE_REF: str(BUNDLE_DIR),
    VLLM_WHEELHOUSE_REF: str(VLLM_WHEELHOUSE_DIR),
    QWEN38_MODEL_SOURCE: str(QWEN_MODEL_DIR),
    ANALYZER_MODEL_ID: str(QWEN_MODEL_DIR),
    "qwen3.8-27B": str(QWEN_MODEL_DIR),
    "duck-qwen3-8-27b-fp8": str(QWEN_MODEL_DIR),
    LEGACY_TAAF_MODEL_REF: str(QWEN_MODEL_DIR),
    f"competition:{REQUIRED_COMPETITION}": str(ARC_COMPETITION_ROOT),
}
setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "ARC_AGI3_COMPETITION_ROOT": str(ARC_COMPETITION_ROOT),
    "ARC_AGI3_WHEELS_DIR": str(ARC_WHEELS_DIR),
    "ARC_AGI3_ENVIRONMENTS_DIR": str(ARC_ENVIRONMENTS_DIR),
    "TAAF_VLLM_WHEELHOUSE": str(VLLM_WHEELHOUSE_DIR),
    "TAAF_QWEN_MODEL_DIR": str(QWEN_MODEL_DIR),
    "ADLDB_QWEN38_MODEL_DIR": str(QWEN_MODEL_DIR),
    "INFERENCE_ANALYZER_MODEL": ANALYZER_MODEL_ID,
    "LOCAL_ANALYZER_MODEL_ID": ANALYZER_MODEL_ID,
    "ARC3_CONTROL_SEED": str(CONTROL_SEED),
    "VLLM_SEED": str(CONTROL_SEED),
    "ARC3_FRAME_MODE": "full",
    "ARC3_STATE_GRAPH": "off",
}
os.environ.update({str(k): str(v) for k, v in setup_env.items()})
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")

AUTO_INPUT_MANIFEST = {
    "schema": "arc3.autoload.inputs.qwen38.v3",
    "competition": {"slug": REQUIRED_COMPETITION, "root": str(ARC_COMPETITION_ROOT), "wheels": str(ARC_WHEELS_DIR), "environment_files": str(ARC_ENVIRONMENTS_DIR)},
    "taaf_source": {"ref": TAAF_SOURCE_REF, "root": str(BUNDLE_DIR)},
    "vllm_wheelhouse": {"ref": VLLM_WHEELHOUSE_REF, "root": str(VLLM_WHEELHOUSE_DIR), "requirements_lock": str(REQUIREMENTS_LOCK), "wheel_count": len(wheel_files)},
    "qwen38": {"model_source": QWEN38_MODEL_SOURCE, "root": str(QWEN_MODEL_DIR), "expected_mount": str(QWEN38_EXPECTED_MOUNT), "weight_shards": len(QWEN_WEIGHT_FILES), "weight_bytes": QWEN_TOTAL_BYTES, "identity_score": QWEN_IDENTITY_SCORE, "model_type": QWEN_MODEL_CONFIG.get("model_type"), "architectures": QWEN_MODEL_CONFIG.get("architectures")},
    "legacy_qwen36_alias_only": {"key": LEGACY_TAAF_MODEL_REF, "resolves_to": str(QWEN_MODEL_DIR)},
    "seed": CONTROL_SEED,
    "expected_served_model": ANALYZER_MODEL_ID,
    "true_submission": bool(TRUE_SUBMISSION),
    "internet_required": False,
}
AUTO_INPUT_MANIFEST_PATH.write_text(json.dumps(AUTO_INPUT_MANIFEST, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print("=" * 96)
print("AUTOLOAD STAGE 1 PASS — ALL REQUIRED INPUTS + QWEN3.8 MODEL RESOLVED + VALIDATED")
print(f"competition       : {ARC_COMPETITION_ROOT}")
print(f"ARC wheels        : {ARC_WHEELS_DIR}")
print(f"TAAF source       : {BUNDLE_DIR}")
print(f"vLLM wheelhouse   : {VLLM_WHEELHOUSE_DIR} ({len(wheel_files)} wheels)")
print(f"Qwen3.8 model     : {QWEN_MODEL_DIR}")
print(f"Qwen3.8 source    : {QWEN38_MODEL_SOURCE}")
print(f"Qwen3.8 shards    : {len(QWEN_WEIGHT_FILES)} / {QWEN_TOTAL_BYTES / 1024**3:.2f} GiB")
print(f"served model ID   : {ANALYZER_MODEL_ID}")
print(f"input manifest    : {AUTO_INPUT_MANIFEST_PATH}")
print("=" * 96)


In [ ]:
# === AUTOLOAD STAGE 2 — INSTALL OFFICIAL ARC RUNTIME OFFLINE ===
# The competition input ships the ARC wheels. Never go to PyPI.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
        "--find-links",
        str(ARC_WHEELS_DIR),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)
import arc_agi as _arc_agi_smoke
print(f"AUTOLOAD STAGE 2 PASS — arc_agi={getattr(_arc_agi_smoke, '__version__', 'unknown')} from {getattr(_arc_agi_smoke, '__file__', None)}")


## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.


In [ ]:
# === PUBLISH RESOLVED KAGGLE INPUTS TO TAAF ===
# Input resolution already ran before installation; this cell exposes those
# validated mounts using the interface expected by the source bundle.
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env.update(
    {
        "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
        "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
        "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    }
)
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(
    json.dumps(setup_env, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.


In [ ]:
# === AUTOLOAD STAGE 4 — LOAD BUNDLED SOURCE + OFFLINE SETUP + RUNTIME AUDIT ===
# Important: do NOT recursively install pyproject dependencies. The known-safe
# execution contract is: bundled source on PYTHONPATH + official setup_commands
# + pinned wheelhouse runtime + explicit smoke checks.
import importlib
import importlib.metadata as _metadata
import re as _re


def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate.resolve())
    return entries


def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    if SETUP_ENV_PATH.is_file():
        env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")

# Validate setup path before executing it. Network/VCS bootstrap is not allowed.
setup_commands = json.loads((BUNDLE_DIR / "setup_commands.json").read_text(encoding="utf-8"))
if not isinstance(setup_commands, list) or not setup_commands:
    raise RuntimeError("TAAF setup_commands.json is empty or invalid")
for command in setup_commands:
    low = str(command).lower()
    if any(token in low for token in ("git clone", "git fetch", "wget http", "curl http")):
        raise RuntimeError(f"NETWORK/VCS SETUP COMMAND REFUSED: {command}")

# Official bundled setup is the only setup path. It consumes the exact canonical
# TAAF_KAGGLE_INPUT_PATHS mapping already written above.
env = _command_env()

def _rewrite_setup_command(command: str) -> str:
    patched = str(command)
    # Immutable Duck/TAAF bundles can hard-code the historical Qwen3.6 served
    # name. The lookup path is already aliased to Qwen3.8; rewrite the served
    # model name too so /v1/models exposes the exact required Qwen3.8 ID.
    patched = patched.replace("vrfai/Qwen3.6-27B-FP8", ANALYZER_MODEL_ID)
    patched = patched.replace("Qwen3.6-27B-FP8", "Qwen3.8-27B-FP8")
    return patched

for raw_command in setup_commands:
    command = _rewrite_setup_command(raw_command)
    if command != raw_command:
        print("AUTOLOAD setup: rewrote legacy Qwen3.6 served-name to Qwen3.8", flush=True)
    print(f"AUTOLOAD setup: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    env = _command_env()
    os.environ.update(env)

# Re-publish source paths and any setup-exported PYTHONPATH.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# The TAAF setup must create this isolated target from requirements.lock.
VLLM_SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"
if not VLLM_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(
        f"AUTOLOAD pinned vLLM site-packages target missing after official setup: {VLLM_SITE_PACKAGES}"
    )
if str(VLLM_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VLLM_SITE_PACKAGES))
    pp = [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
    if str(VLLM_SITE_PACKAGES) not in pp:
        os.environ["PYTHONPATH"] = os.pathsep.join([str(VLLM_SITE_PACKAGES), *pp])

# Explicit notebook/output dependencies only. No recursive TAAF project scanner.
def _pip_install_offline(specs):
    if not specs:
        return
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        "--disable-pip-version-check",
        "--no-warn-conflicts",
        "--find-links", str(VLLM_WHEELHOUSE_DIR),
        "--find-links", str(ARC_WHEELS_DIR),
    ]
    if VLLM_SITE_PACKAGES.is_dir():
        cmd += ["--target", str(VLLM_SITE_PACKAGES), "--upgrade"]
    cmd += list(specs)
    subprocess.check_call(cmd)

required_imports = [
    "arc_agi", "numpy", "pandas", "pyarrow", "torch", "packaging",
    "vllm",
    "inference.agent.action_names", "inference.framework.solver",
    "inference.agent.tool_agent", "taaf.game_api",
]
module_to_dist = {"pyarrow":"pyarrow", "pandas":"pandas", "numpy":"numpy", "packaging":"packaging"}
missing_dists = []
for module_name, dist_name in module_to_dist.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing_dists.append(dist_name)
_pip_install_offline(missing_dists)

# Verify every later-used import now, before any game exists.
import_audit = {}
for module_name in required_imports:
    try:
        module = importlib.import_module(module_name)
    except Exception as exc:
        raise RuntimeError(f"AUTOLOAD REQUIRED IMPORT FAILED: {module_name}: {exc}") from exc
    import_audit[module_name] = {"file": str(getattr(module, "__file__", None)), "version": str(getattr(module, "__version__", None))}

# GPU guard. Do not silently fall back to CPU.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("AUTOLOAD GPU REQUIRED: CUDA is not available")
gpu = torch.cuda.get_device_properties(0)
gpu_name = str(gpu.name)
gpu_gib = float(gpu.total_memory) / 1024**3
if gpu_gib < 70.0:
    raise RuntimeError(f"AUTOLOAD GPU MEMORY TOO SMALL for 27B FP8 control stack: {gpu_name} {gpu_gib:.1f} GiB")

# Validate the pinned vLLM lock if setup created the isolated target. We do not
# install recursively; this only detects missing/wrong pinned runtime packages.
lock_failures = []
lock_entries = 0
try:
    from packaging.requirements import Requirement
    from packaging.markers import default_environment
    target_versions = {}
    if VLLM_SITE_PACKAGES.is_dir():
        for dist in _metadata.distributions(path=[str(VLLM_SITE_PACKAGES)]):
            name = dist.metadata.get("Name")
            if name:
                target_versions[_re.sub(r"[-_.]+", "-", name).lower()] = str(dist.version)
    for raw in REQUIREMENTS_LOCK.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or line.startswith(("-r ", "--requirement ", "-c ", "--constraint ")):
            continue
        try:
            req = Requirement(line)
        except Exception:
            continue
        if req.marker is not None and not req.marker.evaluate(default_environment()):
            continue
        lock_entries += 1
        if target_versions:
            key = _re.sub(r"[-_.]+", "-", req.name).lower()
            installed = target_versions.get(key)
            if installed is None or (req.specifier and installed not in req.specifier):
                lock_failures.append({"requirement": line, "installed": installed})
except Exception as exc:
    raise RuntimeError(f"AUTOLOAD requirements.lock audit failed: {exc}") from exc
if lock_failures:
    raise RuntimeError(f"AUTOLOAD pinned vLLM runtime mismatch: {lock_failures[:25]}")

# Keep control-generation settings bounded exactly as intended.
score_runtime_env = {
    "LOCAL_ANALYZER_MAX_OUTPUT": os.environ.get("TAAF_MAX_OUTPUT_TOKENS", "8192"),
    "LOCAL_ANALYZER_TOOL_STEPS": os.environ.get("TAAF_TOOL_STEPS", "8"),
    "LOCAL_ANALYZER_TEMPERATURE": os.environ.get("TAAF_TEMPERATURE", "0.6"),
    "LOCAL_ANALYZER_TOP_P": os.environ.get("TAAF_TOP_P", "0.95"),
}
os.environ.update(score_runtime_env)
persisted = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
persisted.update(score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(persisted, indent=2, sort_keys=True) + "\n", encoding="utf-8")

# Verify local vLLM server identity and one real completion before ARC gameplay.
def _get_json(url, timeout=10):
    from urllib.request import urlopen
    with urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def _post_json(url, payload, timeout=45):
    from urllib.request import Request, urlopen
    body = json.dumps(payload).encode("utf-8")
    req = Request(url, data=body, headers={"Content-Type":"application/json"}, method="POST")
    with urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def _wait_for_model_server(base_url, timeout_s=300):
    deadline = time.monotonic() + timeout_s
    last = None
    endpoint = base_url.rstrip("/") + "/models"
    while time.monotonic() < deadline:
        try:
            payload = _get_json(endpoint, timeout=10)
            models = payload.get("data") or []
            if models and models[0].get("id"):
                return str(models[0]["id"]), payload
        except Exception as exc:
            last = repr(exc)
        time.sleep(2)
    raise RuntimeError(f"AUTOLOAD vLLM server never became ready at {endpoint}: {last}")

base_url = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1")
served_model_id, models_payload = _wait_for_model_server(base_url)
served_lower = served_model_id.lower()
expected_exact = served_model_id == ANALYZER_MODEL_ID
if not expected_exact:
    raise RuntimeError(
        f"AUTOLOAD WRONG MODEL SERVED: expected exact {ANALYZER_MODEL_ID!r}, "
        f"got {served_model_id!r}. Qwen3.6/path-name fallbacks are disabled."
    )

completion = _post_json(
    base_url.rstrip("/") + "/chat/completions",
    {
        "model": served_model_id,
        "messages": [{"role":"user", "content":"Reply with OK"}],
        "temperature": 0.0,
        "max_tokens": 4,
        "stream": False,
    },
    timeout=60,
)
choices = completion.get("choices") or []
if not choices or not isinstance(choices[0], dict):
    raise RuntimeError(f"AUTOLOAD chat-completion smoke test returned invalid payload: {completion}")

# Normalize the analyzer model to exactly what the running server exposes.
os.environ["INFERENCE_ANALYZER_MODEL"] = served_model_id
os.environ["LOCAL_ANALYZER_MODEL_ID"] = served_model_id
persisted = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
persisted.update({"INFERENCE_ANALYZER_MODEL":served_model_id, "LOCAL_ANALYZER_MODEL_ID":served_model_id})
SETUP_ENV_PATH.write_text(json.dumps(persisted, indent=2, sort_keys=True) + "\n", encoding="utf-8")

AUTO_RUNTIME_AUDIT_PATH = WORKING_DIR / "auto_runtime_audit.json"
AUTO_RUNTIME_AUDIT = {
    "schema":"arc3.autoload.runtime.qwen38.v3",
    "source_roots":[str(x) for x in source_entries],
    "setup_commands":len(setup_commands),
    "recursive_project_dependency_install":False,
    "network_dependency_fetches":0,
    "required_imports":import_audit,
    "vllm_site_packages":str(VLLM_SITE_PACKAGES),
    "requirements_lock_entries_checked":lock_entries,
    "requirements_lock_failures":lock_failures,
    "gpu":{"name":gpu_name, "memory_gib":gpu_gib},
    "resolved_qwen_model_dir":str(QWEN_MODEL_DIR),
    "served_model_id":served_model_id,
    "model_endpoint":base_url,
    "completion_smoke_pass":True,
    "control_seed":CONTROL_SEED,
}
AUTO_RUNTIME_AUDIT_PATH.write_text(json.dumps(AUTO_RUNTIME_AUDIT, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print("=" * 96)
print("AUTOLOAD STAGE 4 PASS — SOURCE + DEPENDENCIES + GPU + MODEL SERVER READY")
print(f"GPU               : {gpu_name} ({gpu_gib:.1f} GiB)")
print(f"source roots      : {len(source_entries)}")
print(f"runtime imports   : {len(import_audit)}/{len(required_imports)}")
print(f"vLLM lock checked : {lock_entries} entries")
print(f"served model      : {served_model_id}")
print(f"completion smoke  : PASS")
print(f"runtime audit     : {AUTO_RUNTIME_AUDIT_PATH}")
print("=" * 96)


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [ ]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 No-external-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only solver-observed actions, rewards, transitions, structured score history, and validated successful trajectories may influence scheduling; hidden labels and external solved paths remain forbidden.


In [ ]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
        "solver_observed_structured_score_history",
        "solver_observed_successful_trajectories",
        "score_validated_cross_game_rule_utility",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: no-external-prior contract active; solver-owned score evidence enabled")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


## 6. ADLDB + Difference-Weighted Exploitation configuration

Every discovered game is still run once. DWE computes a **non-binding advisory budget** from current-game evidence; only environment terminal state and wall-clock timeout can stop a game. A successful transition gets a protected exploit window, while repeated no-progress, stalls, and loops force a strategy-class change.


In [ ]:
# === ADLDB / DIFFERENCE-WEIGHTED EXPLOITATION CONFIGURATION ===
STRICT_NO_EXTERNAL_PRIOR = True
TARGET_CONCURRENCY = int(os.environ.get("ARC3_CONTROL_CONCURRENCY", "28"))
TARGET_SCORE_GAMES = 10
TARGET_MIN_GAME_SCORE = 0.40

# No notebook-injected action cap. Environment terminal state and the
# wall-clock timeout remain authoritative for every game, including ls20.
LS20_MAX_MOVES = None
ADVISORY_MAX_MOVES = 320
GLOBAL_UNCAPPED_ACTION_LIMIT = 1_000_000_000
MAX_STALL_ACTIONS = 12
MAX_NO_PROGRESS_ACTIONS = 12
STALL_ESCAPE_WINDOW = 6
STALL_HARD_WINDOW = 12
MIN_OBSERVATION_ACTIONS = 8
SUCCESS_PROTECT_ACTIONS = 18
DWE_STRICT_LOG_COVERAGE = False
GHOSTBRIDGE_PREMOVE_ENABLED = True
GHOSTBRIDGE_PREMOVE_FAIL_CLOSED = True
GHOSTBRIDGE_PREMOVE_MAX_CONTEXT_FACTS = 8
NO_IMPACT_STREAK_FOR_POLICY_CHANGE = 3
NO_IMPACT_STREAK_FOR_STOP = 8
# Statistical HUD-band overlay constants. These must be defined before
# the first action; the previous run raised NameError here and lost POST state.
NO_IMPACT_BAND_WINDOW = 12
NO_IMPACT_BAND_WARMUP = 3
NO_IMPACT_BAND_THRESHOLD = 0.75
MAX_NO_IMPACT_ACTIONS = NO_IMPACT_STREAK_FOR_STOP

# Difference-Weighted Exploitation signals. Positive terms reward causal evidence;
# negative terms penalize wasted trajectories. These are solver-observed score signals.
EXPLOIT_WEIGHTS = {
    "score": 1.50,
    "level_complete": 4.00,
    "progress_velocity": 2.25,
    "novel_state": 1.00,
    "causal_confidence": 1.75,
    "target_proximity": 2.50,
    "stall": -2.00,
    "repeat_loop": -3.50,
    "no_progress": -2.75,
    "no_impact": -4.25,
    "terminal_loss": -4.00,
}

# Game weight answers: "is this game worth more global computation?"
# Strategy weight answers: "is the current local behavior worth repeating?"
GAME_WEIGHT_DECAY = 0.94
STRATEGY_WEIGHT_DECAY = 0.88
GAME_WEIGHT_LIMIT = 12.0
STRATEGY_WEIGHT_LIMIT = 12.0

# Combined-weight -> advisory action budget only; it is never a binding stop.
DWE_BUDGET_TIERS = (
    (6.0, 320),   # HARD_EXPLOIT
    (3.0, 260),   # EXPLOIT
    (1.0, 210),   # CAUTIOUS_EXPLOIT
    (-1.0, 160),  # BALANCED
    (-3.0, 120),  # EXPLORE / policy transition
    (-999.0, 84), # probable stop-loss trajectory
)

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_action_cap = getattr(bm.solver, "max_actions_per_game", None)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY
bm.solver.max_actions_per_game = GLOBAL_UNCAPPED_ACTION_LIMIT

# Preserve the scored Duck control grafts that reduce wasted actions. Recovery is disabled
# because probe-style recovery can spend extra environment actions and damage efficiency.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": False,
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert bm.solver.max_actions_per_game == GLOBAL_UNCAPPED_ACTION_LIMIT
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags

print(
    "3.57 CONTROL RUN CONFIG: "
    f"strict_no_external_prior={STRICT_NO_EXTERNAL_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    f"hard_cap=UNCAPPED all_games=UNCAPPED "
    f"stall={MAX_STALL_ACTIONS} "
    f"no_progress={MAX_NO_PROGRESS_ACTIONS} "
    f"success_protect={SUCCESS_PROTECT_ACTIONS} "
    f"context={_graft_flags['context_window']} "
    f"seed={CONTROL_SEED} frame_mode={os.environ.get('ARC3_FRAME_MODE')} "
    f"state_graph={os.environ.get('ARC3_STATE_GRAPH')} "
    f"source_per_game_budget={_original_game_budget}",
    flush=True,
)
print("DWE WEIGHTS:", json.dumps(EXPLOIT_WEIGHTS, sort_keys=True), flush=True)


In [ ]:
# === SCORE-PROVEN GAME PRIORITIZATION (EVIDENCE ONLY) ===
from dataclasses import asdict, dataclass, field
from math import log1p
from pathlib import Path
from statistics import median
from typing import Any

GAME_PERFORMANCE_PATH = WORKING_DIR / "game_performance.json"
MAX_SCORING_TRAJECTORIES_PER_GAME = 8
RECENT_SCORE_WINDOW = 12
MARGINAL_ACTION_WINDOW = 24
ADEQUATELY_FUNDED_ACTIONS = 48

TIER_POLICY_WEIGHT = {
    "PROVEN_HIGH": 0.45,
    "PROVEN": 0.30,
    "PROMISING": 0.15,
    "UNPROVEN": 0.07,
    "PERSISTENT_ZERO": 0.03,
}
TIER_ADVISORY_BUDGET = {
    "PROVEN_HIGH": 320,
    "PROVEN": 240,
    "PROMISING": 160,
    "UNPROVEN": 96,
    "PERSISTENT_ZERO": 48,
}
TIER_MODE = {
    "PROVEN_HIGH": "DEEP_EXPLOIT",
    "PROVEN": "EXPLOIT",
    "PROMISING": "TARGETED_EXPLORE",
    "UNPROVEN": "DISCOVERY",
    "PERSISTENT_ZERO": "RECOVERY_PROBE",
}
TIER_MULTIPLIER = {
    "PROVEN_HIGH": 3.0,
    "PROVEN": 2.2,
    "PROMISING": 1.3,
    "UNPROVEN": 0.7,
    "PERSISTENT_ZERO": 0.25,
}


@dataclass
class GamePerformance:
    game_id: str
    episodes: int = 0
    positive_episodes: int = 0
    zero_episodes: int = 0
    adequately_funded_zero_episodes: int = 0
    total_score: float = 0.0
    best_score: float = 0.0
    last_score: float = 0.0
    total_actions: int = 0
    actions_in_positive_episodes: int = 0
    levels_completed: int = 0
    progress_events: int = 0
    causal_events: int = 0
    consecutive_zero_episodes: int = 0
    consecutive_positive_episodes: int = 0
    recent_scores: list[float] = field(default_factory=list)


@dataclass
class ScoringTrajectory:
    game_id: str
    episode_id: str
    score: float
    actions: list[Any]
    transition_signatures: list[str]
    progress_moves: list[int]
    final_move: int
    initial_signature: str = ""

    @property
    def score_per_action(self):
        return self.score / max(1, self.final_move)


@dataclass
class FailureMemory:
    attempted_action_families: list[str] = field(default_factory=list)
    no_impact_controls: list[str] = field(default_factory=list)
    tested_coordinate_regions: list[str] = field(default_factory=list)
    causal_dead_ends: list[str] = field(default_factory=list)
    repeated_terminal_traps: list[str] = field(default_factory=list)


def _bounded_add(values, value, limit=64):
    value = str(value)
    if value and value not in values:
        values.append(value)
        del values[:-limit]


def _action_record(action):
    if isinstance(action, dict):
        return {str(k): v for k, v in action.items()}
    result = {"text": str(action)}
    for name in ("action_id", "id", "data", "name"):
        try:
            value = getattr(action, name)
        except Exception:
            continue
        try:
            json.dumps(value)
            result[name] = value
        except TypeError:
            result[name] = str(value)
    return result


def _action_family(action):
    item = _action_record(action)
    return str(item.get("action_id", item.get("id", item.get("text", "unknown"))))


class GamePerformanceStore:
    def __init__(self, path=None):
        self.path = Path(path) if path else None
        self.games = {}
        self.trajectories = {}
        self.failures = {}
        self.score_rules = {}
        self.active = {}
        self.allocation_actions = {tier: 0 for tier in TIER_POLICY_WEIGHT}
        self.reallocation_pool = 0
        self.imported_sources = set()
        self._load()

    def _load(self):
        if self.path is None or not self.path.is_file():
            return
        try:
            raw = json.loads(self.path.read_text(encoding="utf-8"))
            self.games = {
                gid: GamePerformance(**data)
                for gid, data in raw.get("games", {}).items()
            }
            self.trajectories = {
                gid: [ScoringTrajectory(**item) for item in items]
                for gid, items in raw.get("trajectories", {}).items()
            }
            self.failures = {
                gid: FailureMemory(**item)
                for gid, item in raw.get("failures", {}).items()
            }
            self.score_rules = dict(raw.get("score_rules", {}))
            self.imported_sources = set(raw.get("imported_sources", []))
        except Exception as exc:
            print(f"PRIORITY STORE LOAD WARNING: {type(exc).__name__}: {exc}", flush=True)

    def save(self):
        if self.path is None:
            return
        payload = {
            "schema": "arc3.game-performance.v1",
            "games": {gid: asdict(p) for gid, p in self.games.items()},
            "trajectories": {
                gid: [asdict(item) for item in items]
                for gid, items in self.trajectories.items()
            },
            "failures": {gid: asdict(item) for gid, item in self.failures.items()},
            "score_rules": self.score_rules,
            "imported_sources": sorted(self.imported_sources),
        }
        self.path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.path.with_suffix(".tmp")
        temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
        temporary.replace(self.path)

    def performance(self, game_id):
        key = str(game_id or "unknown")
        if key not in self.games:
            self.games[key] = GamePerformance(game_id=key)
        return self.games[key]

    def failure_memory(self, game_id):
        key = str(game_id or "unknown")
        if key not in self.failures:
            self.failures[key] = FailureMemory()
        return self.failures[key]

    def begin_episode(self, game_id, episode_id=None):
        key = str(game_id or "unknown")
        if key in self.active:
            return self.active[key]["episode_id"]
        eid = str(episode_id or f"{key}-{self.performance(key).episodes + 1}")
        self.active[key] = {
            "episode_id": eid,
            "actions": [],
            "signatures": [],
            "initial_signature": "",
            "progress_moves": [],
            "score": 0.0,
            "levels": 0,
            "progress_events": 0,
            "causal_events": 0,
            "marginal": [],
            "trajectory_diverged": False,
        }
        return eid

    def _scales(self):
        means = [p.total_score / p.episodes for p in self.games.values() if p.episodes and p.total_score > 0]
        efficiencies = [p.total_score / p.total_actions for p in self.games.values() if p.total_actions and p.total_score > 0]
        densities = [p.progress_events / p.total_actions for p in self.games.values() if p.total_actions and p.progress_events]
        return (
            median(means) if means else 1.0,
            median(efficiencies) if efficiencies else 0.01,
            median(densities) if densities else 0.02,
        )

    @staticmethod
    def _robust_normalize(value, scale):
        value = max(0.0, float(value))
        scale = max(float(scale), 1e-12)
        return value / (value + scale)

    def priority_value(self, p):
        if p.episodes <= 0:
            return 0.0
        score_scale, efficiency_scale, progress_scale = self._scales()
        positive_rate = p.positive_episodes / p.episodes
        mean_score = p.total_score / p.episodes
        efficiency = p.total_score / p.total_actions if p.total_actions else 0.0
        density = p.progress_events / p.total_actions if p.total_actions else 0.0
        recurrence = min(p.consecutive_positive_episodes, 3) / 3.0
        zero_penalty = min(p.consecutive_zero_episodes, 5) / 5.0
        return (
            0.30 * positive_rate
            + 0.30 * self._robust_normalize(mean_score, score_scale)
            + 0.20 * self._robust_normalize(efficiency, efficiency_scale)
            + 0.10 * self._robust_normalize(density, progress_scale)
            + 0.10 * recurrence
            - 0.20 * zero_penalty
        )

    def tier(self, p):
        positive_rate = p.positive_episodes / p.episodes if p.episodes else 0.0
        score_scale, _, _ = self._scales()
        mean_score = p.total_score / p.episodes if p.episodes else 0.0
        substantial = self._robust_normalize(mean_score, score_scale) >= 0.45
        if p.positive_episodes >= 3 and positive_rate >= 0.60 and (substantial or p.consecutive_positive_episodes >= 2):
            return "PROVEN_HIGH"
        if p.positive_episodes >= 2 and positive_rate >= 0.40:
            return "PROVEN"
        if p.positive_episodes >= 1 or p.levels_completed > 0 or p.progress_events >= 2:
            return "PROMISING"
        if p.adequately_funded_zero_episodes >= 3 and p.progress_events == 0:
            return "PERSISTENT_ZERO"
        return "UNPROVEN"

    def effective_performance(self, game_id):
        p = GamePerformance(**asdict(self.performance(game_id)))
        active = self.active.get(str(game_id or "unknown"))
        if active and (active["score"] > 0 or active["levels"] > 0 or active["progress_events"] > 0):
            p.episodes += 1
            p.positive_episodes += int(active["score"] > 0)
            p.total_score += max(0.0, active["score"])
            p.best_score = max(p.best_score, active["score"])
            p.total_actions += len(active["actions"])
            p.levels_completed += active["levels"]
            p.progress_events += active["progress_events"]
        return p

    def uncertainty(self, p):
        return 1.0 / (1.0 + max(0, p.episodes))

    def scheduler_weight(self, p):
        tier = self.tier(p)
        return max(self.priority_value(p), 0.01) * TIER_MULTIPLIER[tier] + 0.15 * self.uncertainty(p)

    def marginal_value(self, game_id):
        active = self.active.get(str(game_id or "unknown"), {})
        recent = list(active.get("marginal", []))[-MARGINAL_ACTION_WINDOW:]
        return sum(gain for _, gain in recent) / max(1, sum(actions for actions, _ in recent))

    def context(self, game_id):
        p = self.effective_performance(game_id)
        tier = self.tier(p)
        priority = self.priority_value(p)
        active = self.active.get(str(game_id or "unknown"), {})
        marginal = self.marginal_value(game_id)
        budget = TIER_ADVISORY_BUDGET[tier]
        if tier in ("PROVEN_HIGH", "PROVEN") and self.reallocation_pool:
            bonus = min(self.reallocation_pool, max(16, budget // 4))
            budget += bonus
        return {
            "tier": tier,
            "mode": TIER_MODE[tier],
            "priority": priority,
            "scheduler_weight": self.scheduler_weight(p),
            "advisory_budget": budget,
            "hard_cap": None,
            "best_score": p.best_score,
            "mean_score": p.total_score / p.episodes if p.episodes else 0.0,
            "positive_rate": p.positive_episodes / p.episodes if p.episodes else 0.0,
            "score_per_action": p.total_score / p.total_actions if p.total_actions else 0.0,
            "marginal_value": marginal,
            "successful_prefix_length": len(active.get("actions", [])) if active.get("score", 0.0) > 0 else 0,
            "successful_prefix_confidence": min(1.0, max(0.0, priority) + (0.25 if marginal > 0 else 0.0)),
        }

    def record_transition(self, game_id, action, before, after, event):
        key = str(game_id or "unknown")
        self.begin_episode(key)
        active = self.active[key]
        before_sig = str(before.get("core_signature") or before.get("signature") or "")
        after_sig = str(after.get("core_signature") or after.get("signature") or "")
        if not active["initial_signature"]:
            active["initial_signature"] = before_sig
        active["actions"].append(_action_record(action))
        active["signatures"].append(after_sig)
        score_delta = max(0.0, float(event.get("score_delta", 0.0) or 0.0))
        level_delta = max(0, int(event.get("level_delta", 0) or 0))
        reward = max(0.0, float(event.get("reward", 0.0) or 0.0))
        score = max(0.0, float(event.get("after_score", active["score"]) or 0.0))
        meaningful = bool(score_delta > 0 or level_delta > 0 or reward > 0)
        active["score"] = max(active["score"], score)
        active["levels"] += level_delta
        active["progress_events"] += int(meaningful)
        active["causal_events"] += int(bool(event.get("effective_board_changed")) and not event.get("no_impact"))
        active["marginal"].append((1, score_delta))
        active["marginal"] = active["marginal"][-MARGINAL_ACTION_WINDOW:]
        if meaningful:
            active["progress_moves"].append(len(active["actions"]))

        family = _action_family(action)
        rule = self.score_rules.setdefault(family, {"trials": 0, "score_hits": 0, "score_gain": 0.0, "progress_hits": 0})
        rule["trials"] += 1
        rule["score_hits"] += int(score_delta > 0)
        rule["score_gain"] += score_delta
        rule["progress_hits"] += int(meaningful)

        failure = self.failure_memory(key)
        _bounded_add(failure.attempted_action_families, family)
        if event.get("no_impact"):
            _bounded_add(failure.no_impact_controls, family)
        if event.get("loop_signal") and after_sig:
            _bounded_add(failure.causal_dead_ends, after_sig)
        if event.get("lost") and after_sig:
            _bounded_add(failure.repeated_terminal_traps, after_sig)

        prior = self.trajectory_prior(key, before_sig, len(active["actions"]) - 1)
        if prior and prior.get("expected_signature") and after_sig != prior["expected_signature"]:
            active["trajectory_diverged"] = True

        tier = self.context(key)["tier"]
        self.allocation_actions[tier] += 1
        self.save()

    def finish_episode(self, game_id, score=None, actions=None, levels=None):
        key = str(game_id or "unknown")
        active = self.active.pop(key, None)
        if active is None:
            self.begin_episode(key)
            active = self.active.pop(key)
        final_score = max(0.0, float(active["score"] if score is None else score or 0.0))
        total_actions = int(len(active["actions"]) if actions is None else actions or 0)
        final_levels = int(active["levels"] if levels is None else levels or 0)
        p = self.performance(key)
        p.episodes += 1
        p.last_score = final_score
        p.total_score += final_score
        p.best_score = max(p.best_score, final_score)
        p.total_actions += total_actions
        p.levels_completed += final_levels
        p.progress_events += int(active["progress_events"])
        p.causal_events += int(active["causal_events"])
        p.recent_scores = (p.recent_scores + [final_score])[-RECENT_SCORE_WINDOW:]
        if final_score > 0:
            p.positive_episodes += 1
            p.actions_in_positive_episodes += total_actions
            p.consecutive_positive_episodes += 1
            p.consecutive_zero_episodes = 0
            trajectory = ScoringTrajectory(
                game_id=key,
                episode_id=active["episode_id"],
                score=final_score,
                actions=list(active["actions"]),
                transition_signatures=list(active["signatures"]),
                progress_moves=list(active["progress_moves"]),
                final_move=total_actions,
                initial_signature=active["initial_signature"],
            )
            items = self.trajectories.setdefault(key, [])
            items.append(trajectory)
            items.sort(key=lambda t: (t.score, t.score_per_action, -t.final_move), reverse=True)
            del items[MAX_SCORING_TRAJECTORIES_PER_GAME:]
        else:
            p.zero_episodes += 1
            p.consecutive_zero_episodes += 1
            p.consecutive_positive_episodes = 0
            if total_actions >= ADEQUATELY_FUNDED_ACTIONS:
                p.adequately_funded_zero_episodes += 1
        self.save()
        return p

    def trajectory_prior(self, game_id, current_signature, executed_count=0):
        active = self.active.get(str(game_id or "unknown"), {})
        if active.get("trajectory_diverged"):
            return None
        for trajectory in self.trajectories.get(str(game_id or "unknown"), []):
            index = int(executed_count)
            expected_current = trajectory.initial_signature if index == 0 else (
                trajectory.transition_signatures[index - 1]
                if index - 1 < len(trajectory.transition_signatures) else ""
            )
            if expected_current and str(current_signature or "") == expected_current and index < len(trajectory.actions):
                expected_after = trajectory.transition_signatures[index] if index < len(trajectory.transition_signatures) else ""
                return {
                    "episode_id": trajectory.episode_id,
                    "action": trajectory.actions[index],
                    "expected_signature": expected_after,
                    "prefix_length": index,
                    "score": trajectory.score,
                }
        return None

    def release_unused(self, tier, advisory_budget, used):
        unused = max(0, int(advisory_budget) - int(used))
        if tier in ("UNPROVEN", "PERSISTENT_ZERO"):
            self.reallocation_pool += unused
        return unused

    def ranked(self, game_ids=None):
        ids = list(dict.fromkeys(str(x) for x in (game_ids or self.games.keys())))
        rows = []
        for gid in ids:
            p = self.performance(gid)
            rows.append((self.scheduler_weight(p), self.priority_value(p), gid, p, self.tier(p)))
        return sorted(rows, key=lambda row: (row[0], row[1], row[2]), reverse=True)

    def print_ranking(self, game_ids=None):
        print("GAME PERFORMANCE RANKING", flush=True)
        for rank, (_, priority, gid, p, tier) in enumerate(self.ranked(game_ids), 1):
            mean = p.total_score / p.episodes if p.episodes else 0.0
            efficiency = p.total_score / p.total_actions if p.total_actions else 0.0
            print(
                f"RANK {rank:02d} game={gid} tier={tier} episodes={p.episodes} "
                f"positive={p.positive_episodes} best={p.best_score:.6f} mean={mean:.6f} "
                f"score_per_action={efficiency:.6f} actions={p.total_actions} "
                f"progress_events={p.progress_events} priority={priority:.3f}",
                flush=True,
            )

    def print_allocation(self):
        total = sum(self.allocation_actions.values())
        proven = self.allocation_actions["PROVEN_HIGH"] + self.allocation_actions["PROVEN"]
        print("ALLOCATION SUMMARY", flush=True)
        for tier in TIER_POLICY_WEIGHT:
            print(f"actions_spent tier={tier} actions={self.allocation_actions[tier]}", flush=True)
        concentration = proven / total if total else 0.0
        print(f"SCORING CONCENTRATION proven_action_fraction={concentration:.3%} actions={proven}/{total}", flush=True)

    def ingest_structured_records(self, paths):
        """Import only structured records containing both game_id and an actual score field."""
        imported = 0
        for path in paths:
            path = Path(path)
            if not path.is_file() or path.stat().st_size > 10_000_000:
                continue
            source_id = f"{path.resolve()}:{path.stat().st_size}:{path.stat().st_mtime_ns}"
            if source_id in self.imported_sources:
                continue
            try:
                raw = json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                continue
            records = raw if isinstance(raw, list) else raw.get("games", raw.get("results", [])) if isinstance(raw, dict) else []
            if isinstance(records, dict):
                records = [dict(value, game_id=key) for key, value in records.items() if isinstance(value, dict)]
            accepted = 0
            for item in records:
                if not isinstance(item, dict) or not item.get("game_id"):
                    continue
                score_key = next((name for name in ("actual_score", "final_score", "score") if name in item), None)
                if score_key is None:
                    continue
                try:
                    score = float(item[score_key])
                    actions = int(item.get("actions", item.get("total_actions", 0)) or 0)
                    levels = int(item.get("levels", item.get("levels_completed", 0)) or 0)
                except (TypeError, ValueError):
                    continue
                self.begin_episode(str(item["game_id"]), f"import:{path.name}:{accepted}")
                self.finish_episode(str(item["game_id"]), score=score, actions=actions, levels=levels)
                accepted += 1
            if accepted:
                self.imported_sources.add(source_id)
                imported += accepted
        self.save()
        return imported


def _priority_invariants():
    store = GamePerformanceStore()
    proven = GamePerformance("proven", episodes=5, positive_episodes=4, zero_episodes=1, total_score=80.0, best_score=25.0, total_actions=400, consecutive_positive_episodes=2)
    zero = GamePerformance("zero", episodes=5, zero_episodes=5, adequately_funded_zero_episodes=5, total_actions=400, consecutive_zero_episodes=5)
    store.games = {"proven": proven, "zero": zero}
    assert store.scheduler_weight(proven) > 5 * store.scheduler_weight(zero)
    assert store.tier(GamePerformance("u")) == "UNPROVEN"
    assert store.tier(GamePerformance("p", episodes=1, positive_episodes=1, total_score=1, best_score=1, total_actions=20)) == "PROMISING"
    assert store.tier(GamePerformance("p", episodes=3, positive_episodes=2, zero_episodes=1, total_score=2, best_score=1, total_actions=60)) == "PROVEN"
    assert store.tier(GamePerformance("h", episodes=4, positive_episodes=4, total_score=80, best_score=25, total_actions=200, consecutive_positive_episodes=3)) == "PROVEN_HIGH"
    assert store.tier(zero) == "PERSISTENT_ZERO"
    assert store.scheduler_weight(zero) > 0
    efficient = GamePerformance("efficient", episodes=3, positive_episodes=2, zero_episodes=1, total_score=20, best_score=10, total_actions=80)
    wasteful = GamePerformance("wasteful", episodes=3, positive_episodes=2, zero_episodes=1, total_score=20, best_score=10, total_actions=600)
    store.games.update({"efficient": efficient, "wasteful": wasteful})
    assert store.priority_value(efficient) > store.priority_value(wasteful)
    store.trajectories["proven"] = [ScoringTrajectory("proven", "e1", 10.0, [{"action_id": 1}], ["s1"], [1], 1, "s0")]
    assert store.trajectory_prior("proven", "s0", 0)["action"]["action_id"] == 1
    assert store.trajectory_prior("proven", "different", 0) is None
    old_pool = store.reallocation_pool
    assert store.release_unused("PERSISTENT_ZERO", 48, 8) == 40
    assert store.reallocation_pool > old_pool
    assert TIER_ADVISORY_BUDGET["PROVEN_HIGH"] == 320
    assert all(value is None for value in (None,))  # advisory values never define a hard cap


_priority_invariants()
PERFORMANCE_STORE = GamePerformanceStore(GAME_PERFORMANCE_PATH)
_history_roots = [Path(os.environ.get("ARC3_HISTORICAL_RESULTS_DIR", "/nonexistent"))]
_history_files = []
for _root in _history_roots:
    if _root.is_dir():
        _history_files.extend(sorted(_root.glob("*.json")))
        _history_files.extend(sorted(_root.glob("*.jsonl")))
_imported = PERFORMANCE_STORE.ingest_structured_records(_history_files)
print(
    f"SCORE PRIORITY ACTIVE tiers={TIER_POLICY_WEIGHT} trajectories_per_game={MAX_SCORING_TRAJECTORIES_PER_GAME} "
    f"historical_records_imported={_imported} hard_cap=UNCAPPED",
    flush=True,
)


## 7. Closed-loop ADL + Difference-Weighted Exploitation

The model still performs the two-plan `EXPLOIT` vs `EXPLORE` comparison before each action and a post-move ADL update afterward. In addition, this cell installs a **deterministic runtime DWE auditor** at the real `GameAPI` action boundary.

For every committed action the notebook prints:

- `DWE PRE`: prior game/strategy weights, current decision, live budget, stall/no-progress counters.
- `DWE POST`: before/after score and levels, reward, board/state-change evidence, novelty/loop signals, each weighted term, new weights, and the next allocator decision.

The same records are written to `/kaggle/working/dwe_move_events.jsonl`.


In [ ]:
# === GHOSTBRIDGE v5 IMMUTABLE BELIEF-STATE RUNTIME ===
import json as _gbv5_json
import sys as _gbv5_sys

_GBV5_SOURCES = _gbv5_json.loads("{\"__init__.py\": \"from .integration import GhostBridgeV5Runtime\\nfrom .schemas import PlannerPhase, TransitionEnvelope\\n__all__ = ['GhostBridgeV5Runtime','PlannerPhase','TransitionEnvelope']\\n\", \"causal_graph.py\": \"from __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom dataclasses import dataclass, field\\n\\nfrom .schemas import ActionEvidence, TypedEvent\\n\\n\\n@dataclass(slots=True)\\nclass CausalBelief:\\n    cause: str\\n    effect: str\\n    context: str\\n    support_groups: set[str] = field(default_factory=set)\\n    contradiction_groups: set[str] = field(default_factory=set)\\n    evidence_ids: list[str] = field(default_factory=list)\\n\\n    @property\\n    def confidence(self) -> float:\\n        # Correlated pixel/object/topology views share a dependency group and count once.\\n        support = len(self.support_groups)\\n        contradictions = len(self.contradiction_groups)\\n        return (support + 1.0) / (support + contradictions + 3.0)\\n\\n\\nclass CalibratedCausalGraph:\\n    def __init__(self) -> None:\\n        self.beliefs: dict[tuple[str, str, str], CausalBelief] = {}\\n        self.chains: dict[str, list[tuple[str, str]]] = defaultdict(list)\\n\\n    def update(self, action: ActionEvidence, events: tuple[TypedEvent, ...], context: str) -> tuple[str, ...]:\\n        updated = []\\n        cause = f\\\"action:{action.action}:{dict(action.data)}\\\"\\n        grouped: dict[str, list[TypedEvent]] = defaultdict(list)\\n        for event in events:\\n            grouped[event.dependency_group].append(event)\\n        for dependency_group, correlated in grouped.items():\\n            effects = {event.event_type for event in correlated}\\n            for effect in effects:\\n                key = (cause, effect, context)\\n                belief = self.beliefs.setdefault(key, CausalBelief(cause, effect, context))\\n                if effect == \\\"no_observed_effect\\\":\\n                    belief.contradiction_groups.add(dependency_group)\\n                else:\\n                    belief.support_groups.add(dependency_group)\\n                belief.evidence_ids.extend(event.event_id for event in correlated if event.event_type == effect)\\n                updated.extend(event.event_id for event in correlated if event.event_type == effect)\\n            self.chains[context].append((cause, \\\"+\\\".join(sorted(effects))))\\n        return tuple(dict.fromkeys(updated))\\n\\n    def effects_for(self, action: str, context: str, threshold: float = 0.45) -> tuple[CausalBelief, ...]:\\n        return tuple(\\n            belief for (cause, _, candidate_context), belief in self.beliefs.items()\\n            if cause.startswith(f\\\"action:{action}:\\\") and candidate_context == context and belief.confidence >= threshold\\n        )\\n\\n\", \"completeness.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\nfrom .schemas import TransitionEnvelope\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass CompletenessReport:\\n    complete: bool\\n    missing: tuple[str, ...]\\n\\n\\nclass TransitionCompleteness:\\n    REQUIRED = (\\\"pre_observation\\\", \\\"action\\\", \\\"post_observation\\\", \\\"delta\\\", \\\"events\\\", \\\"assessment\\\", \\\"planner_phase\\\", \\\"hashes\\\")\\n\\n    def check(self, envelope: TransitionEnvelope) -> CompletenessReport:\\n        missing = tuple(name for name in self.REQUIRED if getattr(envelope, name, None) in (None, \\\"\\\"))\\n        try: envelope.verify()\\n        except Exception as exc: missing += (f\\\"integrity:{exc}\\\",)\\n        return CompletenessReport(not missing, missing)\\n\\n\", \"counterfactual_engine.py\": \"from __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\nclass CounterfactualEngine:\\n    def compare(self, predictions: dict[Any, dict[str, float]]) -> tuple[Any, Any] | None:\\n        ranked = sorted(predictions, key=lambda action: (-float(predictions[action].get(\\\"information\\\", 0.0)), str(action)))\\n        return tuple(ranked[:2]) if len(ranked) >= 2 else None\\n\\n    def discriminating_action(self, left: dict[Any, set[str]], right: dict[Any, set[str]]) -> Any | None:\\n        candidates = sorted(set(left) | set(right), key=str)\\n        return max(candidates, key=lambda action: len(left.get(action,set()) ^ right.get(action,set())), default=None)\\n\\n\", \"difference_engine.py\": \"from __future__ import annotations\\n\\nfrom collections import Counter\\n\\nfrom .schemas import DeltaBundle, ImmutableObservation, evidence_id\\n\\n\\nclass HierarchicalDifferenceEngine:\\n    def __init__(self, expensive_cell_threshold: int = 12) -> None:\\n        self.expensive_cell_threshold = expensive_cell_threshold\\n\\n    def diff(self, before: ImmutableObservation, after: ImmutableObservation, force_expensive: bool = False) -> DeltaBundle:\\n        changed = []\\n        height = max(len(before.frame), len(after.frame))\\n        width = max(max((len(r) for r in before.frame), default=0), max((len(r) for r in after.frame), default=0))\\n        for y in range(height):\\n            for x in range(width):\\n                old = before.frame[y][x] if y < len(before.frame) and x < len(before.frame[y]) else None\\n                new = after.frame[y][x] if y < len(after.frame) and x < len(after.frame[y]) else None\\n                if old != new:\\n                    changed.append((x, y, old, new))\\n        score_delta = after.score - before.score\\n        level_delta = after.level - before.level\\n        expensive = force_expensive or bool(score_delta or level_delta or len(changed) >= self.expensive_cell_threshold)\\n        object_events = self._objects(changed) if expensive else ()\\n        topology_events = self._topology(before, after, changed) if expensive else ()\\n        dependency = evidence_id(\\\"dependency\\\", {\\\"pre\\\": before.observation_id, \\\"post\\\": after.observation_id})\\n        payload = {\\\"pre\\\": before.observation_id, \\\"post\\\": after.observation_id, \\\"changed\\\": changed, \\\"objects\\\": object_events, \\\"topology\\\": topology_events, \\\"score_delta\\\": score_delta, \\\"level_delta\\\": level_delta, \\\"dependency\\\": dependency, \\\"resolution\\\": \\\"hierarchical\\\" if expensive else \\\"cheap\\\"}\\n        return DeltaBundle(evidence_id(\\\"delta\\\", payload, (before.observation_id, after.observation_id)), before.observation_id, after.observation_id, tuple(changed), tuple(object_events), tuple(topology_events), score_delta, level_delta, dependency, payload[\\\"resolution\\\"])\\n\\n    @staticmethod\\n    def _objects(changed):\\n        removed = Counter(old for _, _, old, new in changed if old is not None and old != new)\\n        added = Counter(new for _, _, old, new in changed if new is not None and old != new)\\n        events = []\\n        for color in sorted(set(removed) | set(added)):\\n            kind = \\\"moved_or_transformed\\\" if removed[color] and added[color] else \\\"appeared\\\" if added[color] else \\\"disappeared\\\"\\n            events.append({\\\"kind\\\": kind, \\\"color\\\": color, \\\"removed\\\": removed[color], \\\"added\\\": added[color]})\\n        return tuple(events)\\n\\n    @staticmethod\\n    def _topology(before, after, changed):\\n        if not changed:\\n            return ()\\n        before_open = sum(v == 0 for row in before.frame for v in row)\\n        after_open = sum(v == 0 for row in after.frame for v in row)\\n        return ({\\\"kind\\\": \\\"reachability_candidate\\\", \\\"open_cell_delta\\\": after_open - before_open},)\\n\\n\", \"environment_twin.py\": \"from __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\nfrom .schemas import PredictionAssessment, TypedEvent, evidence_id\\n\\n\\n@dataclass(slots=True)\\nclass TwinEffect:\\n    trials: int = 0\\n    event_groups: dict[str, set[str]] = field(default_factory=lambda: defaultdict(set))\\n    errors: list[float] = field(default_factory=list)\\n\\n    def probability(self, event_type: str) -> float:\\n        groups = len(self.event_groups.get(event_type, set()))\\n        return (groups + 1.0) / (self.trials + 2.0)\\n\\n\\nclass EnvironmentTwinV5:\\n    def __init__(self) -> None:\\n        self.effects: dict[tuple[str, str], TwinEffect] = defaultdict(TwinEffect)\\n\\n    def predict(self, context: str, action: str) -> dict[str, Any]:\\n        stats = self.effects[(context, action)]\\n        probabilities = {kind: stats.probability(kind) for kind in stats.event_groups}\\n        return {\\\"context\\\": context, \\\"action\\\": action, \\\"event_probabilities\\\": probabilities, \\\"trials\\\": stats.trials, \\\"confidence\\\": stats.trials / (stats.trials + 3.0)}\\n\\n    def assess(self, prediction: dict[str, Any], events: tuple[TypedEvent, ...]) -> PredictionAssessment:\\n        expected = {kind for kind, probability in prediction.get(\\\"event_probabilities\\\", {}).items() if probability >= 0.5}\\n        observed = {event.event_type for event in events}\\n        union = expected | observed\\n        error = 0.0 if not union else 1.0 - len(expected & observed) / len(union)\\n        parents = tuple(event.event_id for event in events)\\n        payload = {\\\"prediction\\\": prediction, \\\"observed\\\": sorted(observed), \\\"error\\\": error}\\n        return PredictionAssessment(evidence_id(\\\"prediction\\\", payload, parents), error, tuple(sorted(expected & observed)), tuple(sorted(expected - observed)), parents)\\n\\n    def update(self, context: str, action: str, events: tuple[TypedEvent, ...], assessment: PredictionAssessment) -> None:\\n        stats = self.effects[(context, action)]\\n        stats.trials += 1\\n        for event in events:\\n            stats.event_groups[event.event_type].add(event.dependency_group)\\n        stats.errors.append(assessment.error)\\n        del stats.errors[:-64]\\n\\n\", \"event_extractor.py\": \"from __future__ import annotations\\n\\nfrom .schemas import ActionEvidence, DeltaBundle, TypedEvent, evidence_id\\n\\n\\nclass EventExtractor:\\n    def extract(self, action: ActionEvidence, delta: DeltaBundle) -> tuple[TypedEvent, ...]:\\n        events = []\\n        base_parents = (action.action_id, delta.delta_id)\\n        if delta.changed_cells:\\n            events.append(self._event(\\\"frame_changed\\\", {\\\"cells\\\": len(delta.changed_cells)}, base_parents, delta))\\n        for payload in delta.object_events:\\n            events.append(self._event(str(payload[\\\"kind\\\"]), payload, base_parents, delta))\\n        for payload in delta.topology_events:\\n            events.append(self._event(str(payload[\\\"kind\\\"]), payload, base_parents, delta))\\n        if delta.score_delta > 0 or delta.level_delta > 0:\\n            events.append(self._event(\\\"reward\\\", {\\\"score_delta\\\": delta.score_delta, \\\"level_delta\\\": delta.level_delta}, base_parents, delta, 1.0))\\n        if not events:\\n            events.append(self._event(\\\"no_observed_effect\\\", {}, base_parents, delta, 0.9))\\n        return tuple(events)\\n\\n    @staticmethod\\n    def _event(kind, payload, parents, delta, confidence=0.75):\\n        body = {\\\"type\\\": kind, \\\"payload\\\": dict(payload), \\\"dependency_group\\\": delta.dependency_group}\\n        return TypedEvent(evidence_id(\\\"event\\\", body, parents), kind, dict(payload), tuple(parents), confidence, delta.dependency_group)\\n\", \"evidence_ledger.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import asdict\\nfrom hashlib import sha256\\nimport json\\nfrom pathlib import Path\\nimport threading\\nfrom typing import Any\\n\\nfrom .schemas import EvidenceRecord, TransitionEnvelope, canonical\\n\\n\\nclass EvidenceLedger:\\n    \\\"\\\"\\\"Append-only evidence store with atomic transition commits.\\\"\\\"\\\"\\n\\n    def __init__(self, path: str | Path | None = None) -> None:\\n        self.path = Path(path) if path else None\\n        self.records: dict[str, EvidenceRecord] = {}\\n        self.transitions: list[TransitionEnvelope] = []\\n        self._lock = threading.RLock()\\n\\n    def append(self, record: EvidenceRecord) -> str:\\n        with self._lock:\\n            existing = self.records.get(record.record_id)\\n            if existing is not None and existing != record:\\n                raise ValueError(f\\\"immutable evidence collision: {record.record_id}\\\")\\n            missing = [parent for parent in record.parents if parent not in self.records and not parent.startswith((\\\"obs-\\\", \\\"act-\\\", \\\"del-\\\", \\\"eve-\\\", \\\"pre-\\\", \\\"tra-\\\"))]\\n            if missing:\\n                raise ValueError(f\\\"unknown evidence parents: {missing}\\\")\\n            self.records[record.record_id] = record\\n            return record.record_id\\n\\n    def commit(self, envelope: TransitionEnvelope) -> str:\\n        envelope.verify()\\n        with self._lock:\\n            if any(item.transition_id == envelope.transition_id for item in self.transitions):\\n                return envelope.transition_id\\n            if self.transitions and self.transitions[-1].game_id == envelope.game_id and self.transitions[-1].step >= envelope.step:\\n                raise ValueError(\\\"non-monotonic transition commit\\\")\\n            self.transitions.append(envelope)\\n            if self.path:\\n                self.path.parent.mkdir(parents=True, exist_ok=True)\\n                line = canonical(envelope.to_dict()) + \\\"\\\\n\\\"\\n                with self.path.open(\\\"a\\\", encoding=\\\"utf-8\\\") as handle:\\n                    handle.write(line)\\n                    handle.flush()\\n            return envelope.transition_id\\n\\n    def replay_digest(self) -> str:\\n        payload = [item.to_dict() for item in self.transitions]\\n        return sha256(canonical(payload).encode()).hexdigest()\\n\\n    @classmethod\\n    def replay(cls, path: str | Path) -> \\\"EvidenceLedger\\\":\\n        from .integration import envelope_from_dict\\n        ledger = cls(path=None)\\n        for line in Path(path).read_text(encoding=\\\"utf-8\\\").splitlines():\\n            if line.strip():\\n                ledger.commit(envelope_from_dict(json.loads(line)))\\n        return ledger\\n\\n\", \"experiment_planner.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Any, Iterable\\n\\nfrom .schemas import PlannerPhase\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass ExperimentDecision:\\n    mode: str\\n    action: Any\\n    utility: float\\n    rationale: str\\n    phase: PlannerPhase\\n\\n\\nclass ExperimentPlanner:\\n    def __init__(self, zero_score_budget: int = 48) -> None:\\n        self.zero_score_budget = zero_score_budget\\n\\n    def phase(self, state: dict[str, Any]) -> PlannerPhase:\\n        if state.get(\\\"prediction_failed\\\"):\\n            return PlannerPhase.RECOVER\\n        if state.get(\\\"verified_macro\\\"):\\n            return PlannerPhase.EXPLOIT_PROVEN_MACROS\\n        if state.get(\\\"reward_path_expanded\\\"):\\n            return PlannerPhase.EXPAND_REWARD_PATH\\n        if state.get(\\\"reward_path_verified\\\"):\\n            return PlannerPhase.VERIFY_REWARD_PATH\\n        if state.get(\\\"score_seen\\\") and state.get(\\\"backtraced\\\"):\\n            return PlannerPhase.VERIFY_REWARD_PATH\\n        if state.get(\\\"score_seen\\\"):\\n            return PlannerPhase.BACKTRACE_REWARD\\n        if state.get(\\\"mechanics\\\"):\\n            return PlannerPhase.FIRST_SCORE\\n        if state.get(\\\"objects\\\"):\\n            return PlannerPhase.DISCOVER_MECHANICS\\n        if state.get(\\\"controls\\\"):\\n            return PlannerPhase.DISCOVER_OBJECTS\\n        return PlannerPhase.MAP_CONTROLS\\n\\n    def select(self, actions: Iterable[Any], predictions: dict[Any, dict[str, float]], state: dict[str, Any]) -> ExperimentDecision:\\n        actions = tuple(actions)\\n        if not actions:\\n            raise RuntimeError(\\\"no legal experiment actions\\\")\\n        phase = self.phase(state)\\n        scored = []\\n        for action in actions:\\n            prediction = predictions.get(action, {})\\n            reward = float(prediction.get(\\\"reward\\\", 0.0))\\n            information = float(prediction.get(\\\"information\\\", 0.5))\\n            risk = float(prediction.get(\\\"risk\\\", 0.0))\\n            if state.get(\\\"score_seen\\\"):\\n                utility = 4.0 * reward + 0.8 * information - 2.0 * risk\\n            else:\\n                utility = 1.2 * reward + 2.0 * information - 1.5 * risk\\n            scored.append((utility, str(action), action))\\n        utility, _, action = max(scored)\\n        mode = \\\"recover\\\" if phase == PlannerPhase.RECOVER else \\\"exploit\\\" if state.get(\\\"score_seen\\\") else \\\"probe\\\"\\n        return ExperimentDecision(mode, action, utility, f\\\"{phase.value}: expected reward/information/risk\\\", phase)\\n\\n\", \"hypothesis_engine.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom hashlib import sha256\\nfrom typing import Any\\n\\nfrom .schemas import TypedEvent, canonical\\n\\n\\n@dataclass(slots=True)\\nclass MechanicHypothesis:\\n    hypothesis_id: str\\n    claim: str\\n    predicted_events: frozenset[str]\\n    context: str\\n    support_groups: set[str] = field(default_factory=set)\\n    contradiction_groups: set[str] = field(default_factory=set)\\n    evidence_ids: list[str] = field(default_factory=list)\\n    active: bool = True\\n\\n    @property\\n    def confidence(self) -> float:\\n        s, c = len(self.support_groups), len(self.contradiction_groups)\\n        return (s + 1.0) / (s + c + 3.0)\\n\\n\\nclass HypothesisEngine:\\n    def __init__(self) -> None:\\n        self.hypotheses: dict[str, MechanicHypothesis] = {}\\n\\n    def propose(self, claim: str, predicted_events: set[str], context: str) -> MechanicHypothesis:\\n        material = {\\\"claim\\\": claim, \\\"events\\\": sorted(predicted_events), \\\"context\\\": context}\\n        hid = f\\\"hyp-{sha256(canonical(material).encode()).hexdigest()[:20]}\\\"\\n        hypothesis = self.hypotheses.get(hid)\\n        if hypothesis is None:\\n            hypothesis = MechanicHypothesis(hid, claim, frozenset(predicted_events), context)\\n            self.hypotheses[hid] = hypothesis\\n        return hypothesis\\n\\n    def update(self, events: tuple[TypedEvent, ...], context: str) -> tuple[dict[str, Any], ...]:\\n        observed = {event.event_type for event in events}\\n        groups = {event.dependency_group for event in events}\\n        output = []\\n        for hypothesis in self.hypotheses.values():\\n            if not hypothesis.active or hypothesis.context != context:\\n                continue\\n            matched = bool(hypothesis.predicted_events & observed)\\n            target = hypothesis.support_groups if matched else hypothesis.contradiction_groups\\n            target.update(groups)\\n            hypothesis.evidence_ids.extend(event.event_id for event in events)\\n            if len(hypothesis.contradiction_groups) >= 3 and hypothesis.confidence < 0.3:\\n                hypothesis.active = False\\n            output.append({\\\"hypothesis_id\\\": hypothesis.hypothesis_id, \\\"supported\\\": matched, \\\"confidence\\\": hypothesis.confidence, \\\"active\\\": hypothesis.active, \\\"evidence_ids\\\": tuple(event.event_id for event in events)})\\n        return tuple(output)\\n\\n    def competing_pair(self, context: str) -> tuple[MechanicHypothesis, MechanicHypothesis] | None:\\n        active = sorted((h for h in self.hypotheses.values() if h.active and h.context == context), key=lambda h: (-h.confidence, h.hypothesis_id))\\n        for left in active:\\n            for right in active:\\n                if left.hypothesis_id < right.hypothesis_id and left.predicted_events != right.predicted_events:\\n                    return left, right\\n        return None\\n\\n\", \"integration.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import replace\\nfrom hashlib import sha256\\nfrom pathlib import Path\\nimport threading\\nfrom typing import Any, Mapping\\n\\nfrom .causal_graph import CalibratedCausalGraph\\nfrom .completeness import TransitionCompleteness\\nfrom .difference_engine import HierarchicalDifferenceEngine\\nfrom .environment_twin import EnvironmentTwinV5\\nfrom .event_extractor import EventExtractor\\nfrom .evidence_ledger import EvidenceLedger\\nfrom .experiment_planner import ExperimentPlanner\\nfrom .hypothesis_engine import HypothesisEngine\\nfrom .memory_tiers import QuarantinedMemory\\nfrom .object_tracker import PersistentObjectTracker\\nfrom .perception import BudgetedPerception\\nfrom .region_model import RegionModel\\nfrom .reward_path_model import RewardPathModel\\nfrom .schemas import (\\n    ActionEvidence, DeltaBundle, ImmutableObservation, PlannerPhase,\\n    PredictionAssessment, TransitionEnvelope, TypedEvent, canonical, evidence_id,\\n)\\nfrom .temporal_model import TemporalOwnership\\n\\n\\ndef _grid(snapshot: Mapping[str, Any]) -> tuple[tuple[int, ...], ...]:\\n    value = snapshot.get(\\\"grid\\\") or snapshot.get(\\\"frame\\\") or ()\\n    if hasattr(value, \\\"tolist\\\"):\\n        value = value.tolist()\\n    if isinstance(value, (list, tuple)) and value and isinstance(value[0], (list, tuple)):\\n        return tuple(tuple(int(cell) for cell in row) for row in value)\\n    return ()\\n\\n\\ndef _action_parts(action: Any) -> tuple[str, dict[str, Any]]:\\n    if isinstance(action, Mapping):\\n        name = str(action.get(\\\"id\\\", action.get(\\\"action_id\\\", action.get(\\\"name\\\", action))))\\n        data = dict(action.get(\\\"data\\\") or {})\\n        return name, data\\n    name = str(getattr(action, \\\"id\\\", getattr(action, \\\"action_id\\\", getattr(action, \\\"name\\\", action))))\\n    return name, dict(getattr(action, \\\"data\\\", {}) or {})\\n\\n\\ndef _observation(game_id: str, step: int, snapshot: Mapping[str, Any]) -> ImmutableObservation:\\n    return ImmutableObservation.capture(\\n        game_id, step, _grid(snapshot), snapshot.get(\\\"score\\\") or 0.0,\\n        snapshot.get(\\\"levels\\\") or snapshot.get(\\\"level\\\") or 0,\\n        snapshot.get(\\\"game_over\\\") or snapshot.get(\\\"terminal\\\") or False,\\n        {\\\"signature\\\": snapshot.get(\\\"signature\\\"), \\\"won\\\": snapshot.get(\\\"won\\\")},\\n    )\\n\\n\\nclass GhostBridgeV5Runtime:\\n    \\\"\\\"\\\"Production transition boundary: capture, stabilize, explain, predict, commit.\\\"\\\"\\\"\\n\\n    def __init__(self, ledger_path: str | Path | None = None) -> None:\\n        self.ledger = EvidenceLedger(ledger_path)\\n        self.perception = BudgetedPerception(); self.objects = PersistentObjectTracker(); self.regions = RegionModel()\\n        self.difference = HierarchicalDifferenceEngine(); self.events = EventExtractor(); self.causal = CalibratedCausalGraph()\\n        self.hypotheses = HypothesisEngine(); self.twin = EnvironmentTwinV5(); self.reward = RewardPathModel()\\n        self.planner = ExperimentPlanner(); self.memory = QuarantinedMemory(); self.temporal = TemporalOwnership()\\n        self.completeness = TransitionCompleteness(); self.game_state: dict[str, dict[str, Any]] = {}\\n        self.steps: dict[str, int] = {}; self._pending_action: dict[str, ActionEvidence] = {}; self._lock = threading.RLock()\\n\\n    def prepare(self, game_id: str, action: Any, before: Mapping[str, Any]) -> dict[str, Any]:\\n        with self._lock:\\n            step = self.steps.get(game_id, 0) + 1\\n            pre = _observation(game_id, step - 1, before)\\n            name, data = _action_parts(action)\\n            action_record = ActionEvidence.capture(game_id, step, name, data, pre.observation_id)\\n            self.temporal.begin(game_id, step, pre, action_record)\\n            self._pending_action[game_id] = action_record\\n            context = self._context(pre)\\n            return {\\\"game_id\\\": game_id, \\\"step\\\": step, \\\"pre_observation_id\\\": pre.observation_id, \\\"action_id\\\": action_record.action_id, \\\"prediction\\\": self.twin.predict(context, name), \\\"phase\\\": self.planner.phase(self.game_state.get(game_id, {})).value}\\n\\n    def commit(self, game_id: str, after: Mapping[str, Any], intermediate: tuple[Mapping[str, Any], ...] = ()) -> TransitionEnvelope:\\n        with self._lock:\\n            pending = self.temporal.pending.get(game_id)\\n            action = self._pending_action.get(game_id)\\n            if pending is None or action is None:\\n                raise RuntimeError(f\\\"no prepared transition for {game_id}\\\")\\n            snapshots = tuple(intermediate) + (after,)\\n            for index, snapshot in enumerate(snapshots):\\n                observation = _observation(game_id, pending.step, snapshot)\\n                self.temporal.observe(game_id, observation, authoritative=index == len(snapshots) - 1)\\n            post, stabilization_ids = self.temporal.settle(game_id)\\n            pre = pending.pre; context = self._context(pre)\\n            cheap_pre = self.perception.cheap(pre); cheap_post = self.perception.cheap(post)\\n            force_expensive = self.perception.needs_expensive(cheap_pre, cheap_post, post.score - pre.score)\\n            delta = self.difference.diff(pre, post, force_expensive)\\n            events = self.events.extract(action, delta)\\n            prediction = self.twin.predict(context, action.action)\\n            assessment = self.twin.assess(prediction, events)\\n            state = self.game_state.setdefault(game_id, {})\\n            phase = self.planner.phase(state)\\n            hashes = {\\n                \\\"pre\\\": pre.observation_id, \\\"action\\\": action.action_id, \\\"post\\\": post.observation_id,\\n                \\\"delta\\\": delta.delta_id,\\n                \\\"events\\\": sha256(canonical([event.event_id for event in events]).encode()).hexdigest(),\\n            }\\n            material = {\\\"game\\\": game_id, \\\"step\\\": pending.step, **hashes}\\n            transition_id = evidence_id(\\\"transition\\\", material, tuple(hashes.values()))\\n            hypothesis_updates = self.hypotheses.update(events, context)\\n            envelope = TransitionEnvelope(transition_id, game_id, pending.step, pre, action, post, stabilization_ids, delta, events, prediction, assessment, hypothesis_updates, (), phase, hashes)\\n            report = self.completeness.check(envelope)\\n            if not report.complete:\\n                raise RuntimeError(f\\\"transition cannot train model: {report.missing}\\\")\\n            reward_attribution = self.reward.attribute(envelope)\\n            envelope = replace(envelope, reward_attribution=reward_attribution)\\n            envelope.verify()\\n            self.ledger.commit(envelope)\\n            # Model updates occur only after the immutable envelope passes all checks.\\n            self.causal.update(action, events, context); self.twin.update(context, action.action, events, assessment)\\n            self.objects.update(post)\\n            if force_expensive: self.regions.topology(post)\\n            state.update({\\\"controls\\\": True, \\\"objects\\\": force_expensive or state.get(\\\"objects\\\"), \\\"mechanics\\\": bool(self.causal.beliefs), \\\"score_seen\\\": state.get(\\\"score_seen\\\", False) or delta.score_delta > 0 or delta.level_delta > 0, \\\"backtraced\\\": bool(reward_attribution), \\\"prediction_failed\\\": assessment.error > 0.7})\\n            self.memory.observe(game_id, context, f\\\"action:{action.action}\\\", tuple(event.event_type for event in events), delta.score_delta > 0 or delta.level_delta > 0)\\n            self.steps[game_id] = pending.step; del self._pending_action[game_id]\\n            return envelope\\n\\n    @staticmethod\\n    def _context(observation: ImmutableObservation) -> str:\\n        return f\\\"{observation.game_id}|L{observation.level}|{observation.frame_hash[:12]}\\\"\\n\\n    def assert_zero_debt(self, game_id: str) -> None:\\n        if game_id in self.temporal.pending or game_id in self._pending_action:\\n            raise RuntimeError(f\\\"GhostBridge v5 transition debt remains for {game_id}\\\")\\n\\n\\ndef envelope_from_dict(value: Mapping[str, Any]) -> TransitionEnvelope:\\n    def observation(raw): return ImmutableObservation(**raw)\\n    def action(raw): return ActionEvidence(**raw)\\n    def delta(raw):\\n        raw = dict(raw); raw[\\\"changed_cells\\\"] = tuple(tuple(x) for x in raw[\\\"changed_cells\\\"]); raw[\\\"object_events\\\"] = tuple(raw[\\\"object_events\\\"]); raw[\\\"topology_events\\\"] = tuple(raw[\\\"topology_events\\\"])\\n        return DeltaBundle(**raw)\\n    def event(raw): return TypedEvent(**raw)\\n    assessment = PredictionAssessment(**value[\\\"assessment\\\"])\\n    return TransitionEnvelope(value[\\\"transition_id\\\"], value[\\\"game_id\\\"], value[\\\"step\\\"], observation(value[\\\"pre_observation\\\"]), action(value[\\\"action\\\"]), observation(value[\\\"post_observation\\\"]), tuple(value[\\\"stabilization_observations\\\"]), delta(value[\\\"delta\\\"]), tuple(event(x) for x in value[\\\"events\\\"]), value[\\\"prediction\\\"], assessment, tuple(value[\\\"hypothesis_updates\\\"]), tuple(value[\\\"reward_attribution\\\"]), PlannerPhase(value[\\\"planner_phase\\\"]), value[\\\"hashes\\\"])\\n\\n\", \"introspection.py\": \"from __future__ import annotations\\n\\nfrom collections import Counter\\n\\n\\nclass Introspection:\\n    @staticmethod\\n    def snapshot(runtime) -> dict[str, object]:\\n        transitions = runtime.ledger.transitions\\n        return {\\\"transitions\\\": len(transitions), \\\"games\\\": len({x.game_id for x in transitions}), \\\"phases\\\": dict(Counter(x.planner_phase.value for x in transitions)), \\\"ledger_digest\\\": runtime.ledger.replay_digest(), \\\"pending\\\": tuple(sorted(runtime.temporal.pending)), \\\"prediction_models\\\": len(runtime.twin.effects), \\\"hypotheses\\\": len(runtime.hypotheses.hypotheses)}\\n\", \"macro_compiler.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom hashlib import sha256\\nfrom typing import Any, Callable\\n\\nfrom .schemas import canonical\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass GuardedMacro:\\n    macro_id: str\\n    actions: tuple[Any, ...]\\n    preconditions: tuple[Callable[[Any], bool], ...]\\n    postconditions: tuple[Callable[[Any], bool], ...]\\n    evidence_paths: tuple[str, ...]\\n    independent_successes: int\\n\\n\\nclass MacroCompiler:\\n    def compile(self, actions, preconditions, postconditions, evidence_paths, independent_successes: int) -> GuardedMacro:\\n        if independent_successes < 2:\\n            raise ValueError(\\\"reward path must succeed independently twice before macro compilation\\\")\\n        actions = tuple(actions); evidence_paths = tuple(evidence_paths)\\n        material = {\\\"actions\\\": [str(x) for x in actions], \\\"evidence\\\": evidence_paths, \\\"successes\\\": independent_successes}\\n        return GuardedMacro(\\\"mac-\\\" + sha256(canonical(material).encode()).hexdigest()[:20], actions, tuple(preconditions), tuple(postconditions), evidence_paths, independent_successes)\\n\\n\", \"macro_executor.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Any, Callable\\n\\nfrom .macro_compiler import GuardedMacro\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass MacroResult:\\n    completed: bool\\n    actions_executed: int\\n    aborted_reason: str = \\\"\\\"\\n\\n\\nclass MacroExecutor:\\n    def execute(self, macro: GuardedMacro, state: Any, execute_action: Callable[[Any], Any], observe_state: Callable[[], Any]) -> MacroResult:\\n        executed = 0\\n        for index, action in enumerate(macro.actions):\\n            if not all(check(state) for check in macro.preconditions):\\n                return MacroResult(False, executed, f\\\"precondition failed before step {index}\\\")\\n            execute_action(action); executed += 1; state = observe_state()\\n            if index < len(macro.postconditions) and not macro.postconditions[index](state):\\n                return MacroResult(False, executed, f\\\"postcondition failed after step {index}\\\")\\n        return MacroResult(True, executed)\\n\\n\", \"mechanic_registry.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\n\\n\\n@dataclass(slots=True)\\nclass Mechanic:\\n    name: str\\n    preconditions: frozenset[str]\\n    action: str\\n    effects: frozenset[str]\\n    support_contexts: set[str] = field(default_factory=set)\\n    contradiction_contexts: set[str] = field(default_factory=set)\\n\\n    @property\\n    def verified(self) -> bool:\\n        return len(self.support_contexts) >= 2 and not self.contradiction_contexts\\n\\n\\nclass MechanicRegistry:\\n    def __init__(self) -> None:\\n        self.mechanics: dict[str, Mechanic] = {}\\n\\n    def record(self, mechanic: Mechanic) -> None:\\n        self.mechanics[mechanic.name] = mechanic\\n\\n    def verified(self) -> tuple[Mechanic, ...]:\\n        return tuple(item for item in self.mechanics.values() if item.verified)\\n\", \"memory_tiers.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\n\\n@dataclass(slots=True)\\nclass MemoryCandidate:\\n    key: str\\n    value: Any\\n    game_ids: set[str] = field(default_factory=set)\\n    context_ids: set[str] = field(default_factory=set)\\n    successes: int = 0\\n    contradictions: int = 0\\n\\n\\nclass QuarantinedMemory:\\n    def __init__(self, family_games: int = 2, universal_games: int = 3) -> None:\\n        self.family_games = family_games; self.universal_games = universal_games\\n        self.game: dict[str, dict[str, MemoryCandidate]] = {}\\n        self.family: dict[str, MemoryCandidate] = {}\\n        self.universal: dict[str, MemoryCandidate] = {}\\n\\n    def observe(self, game_id: str, context_id: str, key: str, value: Any, success: bool) -> MemoryCandidate:\\n        candidate = self.game.setdefault(game_id, {}).setdefault(key, MemoryCandidate(key, value))\\n        candidate.game_ids.add(game_id); candidate.context_ids.add(context_id)\\n        if success: candidate.successes += 1\\n        else: candidate.contradictions += 1\\n        return candidate\\n\\n    def propose_family(self, family: str, candidates: list[MemoryCandidate]) -> bool:\\n        games = set().union(*(item.game_ids for item in candidates)) if candidates else set()\\n        contexts = set().union(*(item.context_ids for item in candidates)) if candidates else set()\\n        if len(games) < self.family_games or len(contexts) < self.family_games or any(item.contradictions for item in candidates): return False\\n        merged = MemoryCandidate(candidates[0].key, candidates[0].value, games, contexts, sum(x.successes for x in candidates), 0)\\n        self.family[f\\\"{family}:{merged.key}\\\"] = merged; return True\\n\\n    def propose_universal(self, family_keys: list[str]) -> bool:\\n        candidates = [self.family[key] for key in family_keys if key in self.family]\\n        games = set().union(*(item.game_ids for item in candidates)) if candidates else set()\\n        if len(games) < self.universal_games or len(candidates) < 2: return False\\n        key = candidates[0].key\\n        self.universal[key] = MemoryCandidate(key, candidates[0].value, games, set().union(*(x.context_ids for x in candidates)), sum(x.successes for x in candidates), 0)\\n        return True\\n\\n\", \"object_tracker.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom hashlib import sha256\\n\\nfrom .schemas import ImmutableObservation, canonical\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass TrackedObject:\\n    object_id: str\\n    color: int\\n    cells: tuple[tuple[int, int], ...]\\n    parent_ids: tuple[str, ...] = ()\\n\\n\\nclass PersistentObjectTracker:\\n    def __init__(self) -> None:\\n        self.previous: dict[str, tuple[TrackedObject, ...]] = {}\\n\\n    @staticmethod\\n    def _components(observation: ImmutableObservation) -> list[tuple[int, tuple[tuple[int, int], ...]]]:\\n        frame = observation.frame; seen = set(); output = []\\n        for y, row in enumerate(frame):\\n            for x, color in enumerate(row):\\n                if color == 0 or (x, y) in seen: continue\\n                stack = [(x, y)]; seen.add((x, y)); cells = []\\n                while stack:\\n                    cx, cy = stack.pop(); cells.append((cx, cy))\\n                    for nx, ny in ((cx+1,cy),(cx-1,cy),(cx,cy+1),(cx,cy-1)):\\n                        if 0 <= ny < len(frame) and 0 <= nx < len(frame[ny]) and (nx,ny) not in seen and frame[ny][nx] == color:\\n                            seen.add((nx,ny)); stack.append((nx,ny))\\n                output.append((color, tuple(sorted(cells))))\\n        return output\\n\\n    def update(self, observation: ImmutableObservation) -> tuple[TrackedObject, ...]:\\n        prior = self.previous.get(observation.game_id, ())\\n        unmatched = list(prior); tracked = []\\n        for color, cells in self._components(observation):\\n            candidates = [item for item in unmatched if item.color == color]\\n            if candidates:\\n                match = min(candidates, key=lambda item: abs(len(item.cells)-len(cells)) + len(set(item.cells)^set(cells)))\\n                unmatched.remove(match); oid = match.object_id; parents = (match.object_id,)\\n            else:\\n                oid = \\\"obj-\\\" + sha256(canonical((observation.game_id, color, cells)).encode()).hexdigest()[:16]; parents = ()\\n            tracked.append(TrackedObject(oid, color, cells, parents))\\n        result = tuple(tracked); self.previous[observation.game_id] = result\\n        return result\\n\\n\", \"perception.py\": \"from __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Any\\n\\nfrom .schemas import ImmutableObservation\\n\\n\\nclass BudgetedPerception:\\n    def cheap(self, observation: ImmutableObservation) -> dict[str, Any]:\\n        colors = Counter(value for row in observation.frame for value in row)\\n        return {\\\"frame_hash\\\": observation.frame_hash, \\\"shape\\\": (len(observation.frame), max((len(row) for row in observation.frame), default=0)), \\\"palette\\\": tuple(sorted(colors)), \\\"counts\\\": dict(colors)}\\n\\n    def needs_expensive(self, before: dict[str, Any], after: dict[str, Any], score_delta: float = 0.0) -> bool:\\n        return bool(score_delta or before.get(\\\"shape\\\") != after.get(\\\"shape\\\") or before.get(\\\"counts\\\") != after.get(\\\"counts\\\"))\\n\\n\", \"region_model.py\": \"from __future__ import annotations\\n\\nfrom collections import deque\\n\\nfrom .schemas import ImmutableObservation\\n\\n\\nclass RegionModel:\\n    def topology(self, observation: ImmutableObservation) -> dict[str, object]:\\n        open_cells = {(x,y) for y,row in enumerate(observation.frame) for x,value in enumerate(row) if value == 0}\\n        regions = []\\n        while open_cells:\\n            start = next(iter(open_cells)); queue = deque([start]); open_cells.remove(start); region = {start}\\n            while queue:\\n                x,y = queue.popleft()\\n                for neighbor in ((x+1,y),(x-1,y),(x,y+1),(x,y-1)):\\n                    if neighbor in open_cells: open_cells.remove(neighbor); region.add(neighbor); queue.append(neighbor)\\n            regions.append(region)\\n        return {\\\"region_count\\\": len(regions), \\\"region_sizes\\\": tuple(sorted((len(item) for item in regions), reverse=True)), \\\"reachable_cells\\\": sum(map(len, regions))}\\n\\n\", \"reward_path_model.py\": \"from __future__ import annotations\\n\\nfrom collections import defaultdict, deque\\nfrom dataclasses import dataclass\\nfrom typing import Any\\n\\nfrom .schemas import TransitionEnvelope\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass RewardCredit:\\n    transition_id: str\\n    reward_transition_id: str\\n    credit: float\\n    distance: int\\n    evidence_ids: tuple[str, ...]\\n\\n\\nclass RewardPathModel:\\n    def __init__(self, horizon: int = 12) -> None:\\n        self.horizon = horizon\\n        self.history: dict[str, deque[TransitionEnvelope]] = defaultdict(lambda: deque(maxlen=horizon))\\n        self.credits: dict[str, float] = defaultdict(float)\\n        self.verified_paths: dict[str, list[tuple[str, ...]]] = defaultdict(list)\\n\\n    def attribute(self, envelope: TransitionEnvelope) -> tuple[dict[str, Any], ...]:\\n        history = self.history[envelope.game_id]\\n        output = []\\n        reward = envelope.delta.score_delta + max(0, envelope.delta.level_delta)\\n        if reward > 0:\\n            chain = list(history) + [envelope]\\n            path = tuple(item.action.action for item in chain)\\n            for distance, item in enumerate(reversed(chain)):\\n                credit = reward * (0.72 ** distance)\\n                self.credits[item.transition_id] += credit\\n                output.append({\\\"transition_id\\\": item.transition_id, \\\"reward_transition_id\\\": envelope.transition_id, \\\"credit\\\": credit, \\\"distance\\\": distance, \\\"evidence_ids\\\": (item.delta.delta_id, envelope.delta.delta_id)})\\n            if path not in self.verified_paths[envelope.game_id]:\\n                self.verified_paths[envelope.game_id].append(path)\\n        history.append(envelope)\\n        return tuple(output)\\n\\n    def expected_additional_reward(self, game_id: str) -> float:\\n        paths = self.verified_paths.get(game_id, ())\\n        return sum(1.0 / max(1, len(path)) for path in paths)\\n\\n\", \"schemas.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom enum import Enum\\nfrom hashlib import sha256\\nimport json\\nfrom typing import Any, Mapping\\n\\n\\ndef canonical(value: Any) -> str:\\n    return json.dumps(value, sort_keys=True, separators=(\\\",\\\", \\\":\\\"), default=str)\\n\\n\\ndef evidence_id(kind: str, payload: Any, parents: tuple[str, ...] = ()) -> str:\\n    material = canonical({\\\"kind\\\": kind, \\\"parents\\\": parents, \\\"payload\\\": payload})\\n    return f\\\"{kind[:3]}-{sha256(material.encode()).hexdigest()[:24]}\\\"\\n\\n\\nclass PlannerPhase(str, Enum):\\n    MAP_CONTROLS = \\\"MAP_CONTROLS\\\"\\n    DISCOVER_OBJECTS = \\\"DISCOVER_OBJECTS\\\"\\n    DISCOVER_MECHANICS = \\\"DISCOVER_MECHANICS\\\"\\n    FIRST_SCORE = \\\"FIRST_SCORE\\\"\\n    BACKTRACE_REWARD = \\\"BACKTRACE_REWARD\\\"\\n    VERIFY_REWARD_PATH = \\\"VERIFY_REWARD_PATH\\\"\\n    EXPAND_REWARD_PATH = \\\"EXPAND_REWARD_PATH\\\"\\n    EXPLOIT_PROVEN_MACROS = \\\"EXPLOIT_PROVEN_MACROS\\\"\\n    RECOVER = \\\"RECOVER\\\"\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass EvidenceRecord:\\n    record_id: str\\n    kind: str\\n    payload: Mapping[str, Any]\\n    parents: tuple[str, ...] = ()\\n    game_id: str = \\\"\\\"\\n    transition_id: str = \\\"\\\"\\n\\n    @classmethod\\n    def create(cls, kind: str, payload: Mapping[str, Any], parents=(), game_id=\\\"\\\", transition_id=\\\"\\\"):\\n        parents = tuple(parents)\\n        return cls(evidence_id(kind, payload, parents), kind, dict(payload), parents, game_id, transition_id)\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass ImmutableObservation:\\n    observation_id: str\\n    game_id: str\\n    step: int\\n    frame_hash: str\\n    frame: tuple[tuple[int, ...], ...]\\n    score: float\\n    level: int\\n    terminal: bool\\n    metadata: Mapping[str, Any] = field(default_factory=dict)\\n\\n    @classmethod\\n    def capture(cls, game_id: str, step: int, frame, score=0.0, level=0, terminal=False, metadata=None):\\n        frozen = tuple(tuple(int(v) for v in row) for row in (frame or ()))\\n        payload = {\\\"game_id\\\": game_id, \\\"step\\\": step, \\\"frame\\\": frozen, \\\"score\\\": float(score), \\\"level\\\": int(level), \\\"terminal\\\": bool(terminal), \\\"metadata\\\": dict(metadata or {})}\\n        return cls(evidence_id(\\\"observation\\\", payload), game_id, step, sha256(canonical(frozen).encode()).hexdigest(), frozen, float(score), int(level), bool(terminal), dict(metadata or {}))\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass ActionEvidence:\\n    action_id: str\\n    game_id: str\\n    step: int\\n    action: str\\n    data: Mapping[str, Any]\\n    pre_observation_id: str\\n\\n    @classmethod\\n    def capture(cls, game_id: str, step: int, action: Any, data: Mapping[str, Any], pre_observation_id: str):\\n        payload = {\\\"game_id\\\": game_id, \\\"step\\\": step, \\\"action\\\": str(action), \\\"data\\\": dict(data), \\\"pre\\\": pre_observation_id}\\n        return cls(evidence_id(\\\"action\\\", payload, (pre_observation_id,)), game_id, step, str(action), dict(data), pre_observation_id)\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass DeltaBundle:\\n    delta_id: str\\n    pre_observation_id: str\\n    post_observation_id: str\\n    changed_cells: tuple[tuple[int, int, int | None, int | None], ...]\\n    object_events: tuple[Mapping[str, Any], ...]\\n    topology_events: tuple[Mapping[str, Any], ...]\\n    score_delta: float\\n    level_delta: int\\n    dependency_group: str\\n    resolution: str\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass TypedEvent:\\n    event_id: str\\n    event_type: str\\n    payload: Mapping[str, Any]\\n    parents: tuple[str, ...]\\n    confidence: float\\n    dependency_group: str\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass PredictionAssessment:\\n    prediction_id: str\\n    error: float\\n    matched: tuple[str, ...]\\n    contradicted: tuple[str, ...]\\n    parents: tuple[str, ...]\\n\\n\\n@dataclass(frozen=True, slots=True)\\nclass TransitionEnvelope:\\n    transition_id: str\\n    game_id: str\\n    step: int\\n    pre_observation: ImmutableObservation\\n    action: ActionEvidence\\n    post_observation: ImmutableObservation\\n    stabilization_observations: tuple[str, ...]\\n    delta: DeltaBundle\\n    events: tuple[TypedEvent, ...]\\n    prediction: Mapping[str, Any]\\n    assessment: PredictionAssessment\\n    hypothesis_updates: tuple[Mapping[str, Any], ...]\\n    reward_attribution: tuple[Mapping[str, Any], ...]\\n    planner_phase: PlannerPhase\\n    hashes: Mapping[str, str]\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        value = asdict(self)\\n        value[\\\"planner_phase\\\"] = self.planner_phase.value\\n        return value\\n\\n    def verify(self) -> None:\\n        if self.action.pre_observation_id != self.pre_observation.observation_id:\\n            raise ValueError(\\\"action/pre-observation provenance mismatch\\\")\\n        if self.delta.pre_observation_id != self.pre_observation.observation_id or self.delta.post_observation_id != self.post_observation.observation_id:\\n            raise ValueError(\\\"delta observation provenance mismatch\\\")\\n        expected = {\\n            \\\"pre\\\": self.pre_observation.observation_id,\\n            \\\"action\\\": self.action.action_id,\\n            \\\"post\\\": self.post_observation.observation_id,\\n            \\\"delta\\\": self.delta.delta_id,\\n            \\\"events\\\": sha256(canonical([e.event_id for e in self.events]).encode()).hexdigest(),\\n        }\\n        if dict(self.hashes) != expected:\\n            raise ValueError(\\\"transition evidence hashes disagree\\\")\\n        material = {\\\"game\\\": self.game_id, \\\"step\\\": self.step, **expected}\\n        if self.transition_id != evidence_id(\\\"transition\\\", material, tuple(expected.values())):\\n            raise ValueError(\\\"transition envelope identity mismatch\\\")\\n\\n\", \"temporal_model.py\": \"from __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\nfrom .schemas import ImmutableObservation\\n\\n\\n@dataclass(slots=True)\\nclass PendingAction:\\n    game_id: str\\n    step: int\\n    pre: ImmutableObservation\\n    action: Any\\n    deadline_tick: int\\n    observations: list[ImmutableObservation] = field(default_factory=list)\\n\\n\\nclass TemporalOwnership:\\n    def __init__(self, stable_frames: int = 2, deadline_frames: int = 8) -> None:\\n        self.stable_frames = max(1, stable_frames)\\n        self.deadline_frames = max(self.stable_frames, deadline_frames)\\n        self.pending: dict[str, PendingAction] = {}\\n\\n    def begin(self, game_id: str, step: int, pre: ImmutableObservation, action: Any) -> PendingAction:\\n        if game_id in self.pending:\\n            raise RuntimeError(f\\\"action still pending for {game_id}\\\")\\n        item = PendingAction(game_id, step, pre, action, step + self.deadline_frames)\\n        self.pending[game_id] = item\\n        return item\\n\\n    def observe(self, game_id: str, observation: ImmutableObservation, authoritative: bool = False) -> bool:\\n        item = self.pending[game_id]\\n        item.observations.append(observation)\\n        recent = item.observations[-self.stable_frames:]\\n        stable = len(recent) >= self.stable_frames and len({x.frame_hash for x in recent}) == 1\\n        expired = authoritative or len(item.observations) >= self.deadline_frames or observation.terminal\\n        return stable or expired\\n\\n    def settle(self, game_id: str) -> tuple[ImmutableObservation, tuple[str, ...]]:\\n        item = self.pending.pop(game_id)\\n        if not item.observations:\\n            raise RuntimeError(\\\"cannot settle action without post observation\\\")\\n        return item.observations[-1], tuple(x.observation_id for x in item.observations)\\n\"}")
_GBV5_ROOT = WORKING_DIR / "ghostbridge_v5"
_GBV5_ROOT.mkdir(parents=True, exist_ok=True)
for _gbv5_name, _gbv5_source in _GBV5_SOURCES.items():
    (_GBV5_ROOT / _gbv5_name).write_text(_gbv5_source, encoding="utf-8")
if str(WORKING_DIR) not in _gbv5_sys.path:
    _gbv5_sys.path.insert(0, str(WORKING_DIR))

from ghostbridge_v5.integration import GhostBridgeV5Runtime
from ghostbridge_v5.introspection import Introspection as GhostBridgeV5Introspection

_GBV5_LEDGER_PATH = WORKING_DIR / "ghostbridge_v5_transition_ledger.jsonl"
_GBV5_LEDGER_PATH.unlink(missing_ok=True)
_GBV5_RUNTIME = GhostBridgeV5Runtime(_GBV5_LEDGER_PATH)
print(
    "GHOSTBRIDGE v5 ACTIVE: immutable evidence + stabilization + calibrated causality + "
    "environment twin + reward paths + guarded macros + quarantined memory",
    flush=True,
)


In [ ]:
# === CLOSED-LOOP ADL + DIFFERENCE-WEIGHTED EXPLOITATION ===
import asyncio
import contextvars
import hashlib
import inspect
import json
import math
import os
import sys
import threading
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Mapping
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2
ADL_DEBT_RECOVERY_ENABLED = True
GHOSTBRIDGE_NEGATIVE_SPACE_ENABLED = True

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"
DWE_MOVE_LOG = WORKING_DIR / "dwe_move_events.jsonl"
DWE_SUMMARY_LOG = WORKING_DIR / "dwe_game_summaries.jsonl"
GHOSTBRIDGE_PRE_MOVE_LOG = WORKING_DIR / "ghostbridge_premove_pre_move_events.jsonl"

for _path in (DUAL_PATH_POLICY_LOG, POST_MOVE_ADL_LOG, DWE_MOVE_LOG, DWE_SUMMARY_LOG, GHOSTBRIDGE_PRE_MOVE_LOG):
    try:
        _path.unlink(missing_ok=True)
    except Exception:
        pass

CLOSED_LOOP_ADL_INSTRUCTION = r"""
ADLDB / GHOSTBRIDGE v2 CLOSED-LOOP POLICY

You have exactly ONE real environment trajectory for the current game. Never
fork, clone, reset for speculation, or use another game's state. Use only
observations/actions/rewards/transitions learned in THIS game in THIS run.

INVARIANT 0 — NO UNREPAIRED ADL DEBT
Before planning the next real action, the immediately preceding committed action
must have a POST_MOVE_ADL/DWE_POST record. If the full analyzer update failed,
use the deterministic Python transition snapshot to emit a degraded recovery
record. If even that recovery cannot be persisted, STOP rather than take another
move. Never silently continue with missing post-move learning.

GHOSTBRIDGE_PREMOVE — PRE-MOVE PLANNING + NEGATIVE-SPACE DIRECTOR
GhostBridgePreMove runs BEFORE ADL decides every real move. It is advisory only: it never
steps the environment, never chooses the final action, never forks the game, and
never imports cross-game knowledge. Its job is to write the smallest current-game plan/brief that exposes what ADL is likely missing,
prioritizes action efficiency, and forbids redundant environment probes.

For every move, first emit exactly one compact marker:
GHOSTBRIDGE_PRE_MOVE_PLAN:
STEP=<integer>
KNOWN_CAUSAL=<confirmed current-game cause/effect or none>
MISSING_CAPABILITY=<most plausible absent/disconnected capability>
FALSIFIED=<action/strategy class contradicted by evidence or none>
COUNTERFACTUAL=<what should change if the missing capability is real>
INFO_TARGET=<highest-value uncertainty to resolve next>
CONSTRAINT=<legality/budget/no-repeat constraint>
ADL_GUIDANCE=<short instruction to ADL; do not name the final action>

ADL MUST consume this brief before constructing candidate A and B. GhostBridgePreMove may
change the hypothesis class, information target, or forbidden repeats, but the ADL
layer remains responsible for candidate generation and final selection.

GHOSTBRIDGE — NEGATIVE SPACE LEARNING
Infer missing capabilities from causal absences: no-impact actions, disconnected
controls, repeated unchanged state-action pairs, stalled local policies, missing
interaction hypotheses, and transitions that should have occurred but did not.
Build the smallest current-game bridge that can test or restore the missing
capability. After 6 no-progress/stall actions force a strategy-class change; at
12 reject equivalent exhausted probes and broaden the interaction hypothesis.

BEFORE EVERY REAL ACTION
0. Read and obey the current GHOSTBRIDGE_PRE_MOVE_PLAN.
1. Construct exactly two legal candidate actions from the same current state:
   A = EXPLOIT: shortest move supported by confirmed causal evidence.
   B = EXPLORE: highest-information legal move not already exhausted.
2. Compare legality, predicted progress, predicted frame/state change,
   information gain, loop risk, action cost, and current-game consistency.
3. Maintain two game-local action values plus a separate score-evidence game priority:
   GAME_EXPLOIT_WEIGHT: whether this GAME deserves more computation.
   STRATEGY_EXPLOIT_WEIGHT: whether the CURRENT STRATEGY deserves repetition.
4. Print/record in your reasoning trace before the tool call:

DWE_PRE_DECISION:
STEP=<integer>
GAME_WEIGHT=<number>
STRATEGY_WEIGHT=<number>
A_ACTION=<candidate A>
B_ACTION=<candidate B>
SELECT=<A or B>
MODE=<HARD_EXPLOIT|EXPLOIT|CAUTIOUS_EXPLOIT|BALANCED|EXPLORE|CHANGE_POLICY>
WHY=<solver-observed evidence only>

Then issue exactly ONE real environment action.

IMMEDIATELY AFTER EVERY REAL ACTION
Compare pre-state, prediction, action and actual returned state. Record:

POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<number or unknown>
LEVEL_DELTA=<number or unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0..1>
PROGRESS_VALUE=<-1..1>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game lesson>
NEXT_BIAS=<exploit/explore/change_policy/neutral>

Perception/control principles:
- use the full current frame; treat animation/change as evidence, not decoration;
- optimize level depth and verified score progress;
- a visual change confined to a deterministic HUD/moves band is NO_IMPACT;
- ACTION7 is legal when exposed by the environment;

DWE exploitation principles:
- verified level completion is the strongest positive signal;
- positive score/reward/progress increases both game and strategy value;
- novel useful transitions increase information value;
- repeated unchanged states, loops and no-progress streaks reduce strategy value;
- a promising game with a weak strategy means CHANGE_POLICY, not immediate abandon;
- sustained low game/strategy value means CHANGE_POLICY and continued exploration;
- after verified success, exploit the causal pattern for a protected window;
- never let one lucky early transition permanently monopolize the budget.

The runtime independently audits these decisions and prints DWE PRE / DWE POST
for every actual environment action. DWE never terminates a nonterminal game.
No game has a notebook-injected hard action cap. Use only solver-observed evidence; never fabricate scoring evidence.
""".strip()


class GhostBridgePreMoveADLToolAgent(ToolAgent):
    """Duck ToolAgent whose every ADL turn is prefaced by GhostBridgePreMove guidance."""

    def __init__(self, *args, game_id="unknown", **kwargs):
        super().__init__(*args, **kwargs)
        self._ghostbridge_premove_game_id = str(game_id or "unknown")
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION

    def _build_user_prompt(
        self,
        action_num: int,
        *,
        valid_actions=None,
        current_frame=None,
        history_entries=None,
        previous_step_summary=None,
    ):
        # This hook occurs before the analyzer chooses the next real action.
        # Fail closed on unresolved POST_MOVE_ADL debt, then inject a deterministic
        # current-game GhostBridgePreMove brief into the same Duck-v12 control-model reasoning turn.
        game_id = self._ghostbridge_premove_game_id
        if GHOSTBRIDGE_PREMOVE_ENABLED and "DWE_ALLOCATOR" in globals():
            _ghostbridge_assert_no_adl_debt(game_id)
        base = super()._build_user_prompt(
            action_num,
            valid_actions=valid_actions,
            current_frame=current_frame,
            history_entries=history_entries,
            previous_step_summary=previous_step_summary,
        )
        if not GHOSTBRIDGE_PREMOVE_ENABLED:
            return base
        brief = _ghostbridge_premove_brief(
            game_id=game_id,
            action_num=action_num,
            valid_actions=valid_actions,
            current_frame=current_frame,
            previous_step_summary=previous_step_summary,
        )
        return base + "\n\n" + brief


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or ANALYZER_MODEL_ID
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    kwargs = {
        "model": model,
        "timeout": bm.solver.analyzer_timeout,
        "save_request_logs": bm.solver.save_request_logs,
        "base_url": base_url,
        "provider": "vllm",
    }
    # Preserve compatibility with multiple ToolAgent versions while passing the
    # control seed at request level whenever the installed implementation
    # explicitly supports a seed-bearing argument.
    try:
        base_params = inspect.signature(ToolAgent.__init__).parameters
    except Exception:
        base_params = {}
    if "seed" in base_params:
        kwargs["seed"] = CONTROL_SEED
    elif "request_kwargs" in base_params:
        kwargs["request_kwargs"] = {"seed": CONTROL_SEED}
    elif "model_kwargs" in base_params:
        kwargs["model_kwargs"] = {"seed": CONTROL_SEED}
    try:
        game_id = _extract_game_id(game)
    except Exception:
        game_id = str(getattr(game, "game_id", None) or getattr(game, "env_name", None) or f"game-{index}")
    return GhostBridgePreMoveADLToolAgent(game_id=game_id, **kwargs)


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory


def _clip(value, low, high):
    return max(low, min(high, float(value)))


def _num(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        x = float(value)
        if math.isfinite(x):
            return x
    except Exception:
        pass
    return default


def _read(obj, names, default=None):
    if obj is None:
        return default
    for name in names:
        try:
            if isinstance(obj, Mapping) and name in obj:
                value = obj[name]
            elif hasattr(obj, name):
                value = getattr(obj, name)
            else:
                continue
            if callable(value):
                continue
            if value is not None:
                return value
        except Exception:
            continue
    return default


def _walk_candidates(obj, max_depth=2):
    """Yield a small, safe object graph for score/state field discovery."""
    seen = set()
    queue = deque([(obj, 0)])
    child_names = (
        "state", "game_state", "observation", "result", "info", "metadata",
        "response", "frame", "board", "env", "game", "run",
    )
    while queue:
        cur, depth = queue.popleft()
        if cur is None or id(cur) in seen:
            continue
        seen.add(id(cur))
        yield cur
        if depth >= max_depth:
            continue
        for name in child_names:
            nxt = _read(cur, (name,), None)
            if nxt is not None and not isinstance(nxt, (str, bytes, int, float, bool)):
                queue.append((nxt, depth + 1))


def _find_number(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _num(_read(candidate, names, None), None)
            if value is not None:
                return value
    return default


def _find_bool(objs, names, default=None):
    for obj in objs:
        for candidate in _walk_candidates(obj):
            value = _read(candidate, names, None)
            if isinstance(value, bool):
                return value
            if isinstance(value, (int, float)) and value in (0, 1):
                return bool(value)
            if isinstance(value, str):
                v = value.strip().lower()
                if v in {"true", "yes", "won", "lost", "done", "terminal", "game_over"}:
                    return True
                if v in {"false", "no", "playing", "active", "running"}:
                    return False
    return default


def _extract_game_id(*objs):
    names = ("game_id", "env_name", "environment_id", "game_name", "name")
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            value = _read(candidate, names, None)
            if value is not None:
                text = str(value).strip()
                if text:
                    return text
    return "unknown"


def _extract_action(args, kwargs):
    for key in ("action", "action_name", "action_spec", "move", "command"):
        if key in kwargs:
            return str(kwargs[key])
    for value in args:
        if value is None:
            continue
        text = str(value)
        if text and len(text) <= 300:
            return text
    return "unknown"


def _visual_payload(*objs):
    """Return the first likely 2-D/3-D visual state payload without mutating it."""
    field_names = (
        "board", "grid", "frame", "image", "observation", "pixels",
        "screen", "state_matrix", "board_state",
    )
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=2):
            for name in field_names:
                value = _read(candidate, (name,), None)
                if value is None:
                    continue
                try:
                    if hasattr(value, "tolist"):
                        value = value.tolist()
                    # Require a matrix-like payload; text observations are not
                    # useful for the deterministic HUD-band comparison.
                    if (
                        isinstance(value, (list, tuple))
                        and len(value) >= 3
                        and isinstance(value[0], (list, tuple))
                    ):
                        return value
                except Exception:
                    continue
    return None


def _hash_payload(value):
    if value is None:
        return None
    try:
        payload = json.dumps(
            value,
            sort_keys=True,
            default=str,
            separators=(",", ":"),
        )
    except Exception:
        payload = repr(value)
    if not payload or len(payload) <= 4:
        return None
    return hashlib.sha1(
        payload[:2_000_000].encode("utf-8", "replace")
    ).hexdigest()[:16]


def _core_visual_payload(value):
    """Remove only thin outer HUD/moves bands; retain almost the entire board.

    This is deliberately conservative. The no-impact detector fires only when
    the full frame changes while this core stays identical and there is no
    score/reward/level progress. It therefore cannot manufacture positive
    evidence; it only discounts likely cosmetic/HUD-only changes.
    """
    if value is None or not isinstance(value, (list, tuple)) or len(value) < 8:
        return value
    rows = list(value)
    width = min(
        (len(row) for row in rows if isinstance(row, (list, tuple))),
        default=0,
    )
    if width < 8:
        return value

    # Trim 6.25% from each edge, capped so at least 6x6 content remains.
    trim_y = min(max(1, len(rows) // 16), max(1, (len(rows) - 6) // 2))
    trim_x = min(max(1, width // 16), max(1, (width - 6) // 2))
    core = []
    for row in rows[trim_y:len(rows) - trim_y]:
        if isinstance(row, (list, tuple)):
            core.append(list(row)[trim_x:width - trim_x])
    return core or value


def _stable_signature(*objs):
    return _hash_payload(_visual_payload(*objs))


def _core_signature(*objs):
    visual = _visual_payload(*objs)
    return _hash_payload(_core_visual_payload(visual))

def _snapshot(api, result=None):
    objs = tuple(x for x in (result, api) if x is not None)
    score = _find_number(objs, ("score", "current_score", "total_score", "game_score", "final_score"), None)
    levels = _find_number(objs, ("levels_completed", "level_completed_count", "completed_levels", "level"), None)
    reward = _find_number((result,), ("reward", "score_delta", "delta_reward"), None)
    board_changed = _find_bool((result,), ("board_changed", "state_changed", "frame_changed", "changed"), None)
    level_completed = _find_bool((result,), ("level_completed", "completed_level", "level_won"), None)
    game_over = _find_bool(objs, ("game_over", "done", "terminal", "is_done", "finished"), None)
    won = _find_bool(objs, ("won", "is_won", "victory"), None)
    lost = _find_bool(objs, ("lost", "is_lost", "defeat"), None)
    signature = _stable_signature(result, api)
    core_signature = _core_signature(result, api)
    return {
        "score": score,
        "levels": int(levels) if levels is not None else None,
        "reward": reward,
        "board_changed": board_changed,
        "level_completed": level_completed,
        "game_over": game_over,
        "won": won,
        "lost": lost,
        "signature": signature,
        "core_signature": core_signature,
    }


def _dwe_game_key(game_id):
    value = str(game_id or "").strip().lower()
    return value.split("-", 1)[0] if value else "unknown"

def _is_ls20_game(game_id):
    return _dwe_game_key(game_id) == "ls20"

def _hard_action_cap(game_id):
    return None

def _hard_cap_label(game_id):
    cap = _hard_action_cap(game_id)
    return str(cap) if cap is not None else "UNCAPPED"

@dataclass
class DWEGameState:
    game_id: str
    move: int = 0
    last_score: float = 0.0
    last_levels: int = 0
    game_weight: float = 0.0
    strategy_weight: float = 0.0
    combined_weight: float = 0.0
    progress_velocity: float = 0.0
    stall_streak: int = 0
    no_progress_streak: int = 0
    repeat_streak: int = 0
    no_impact_streak: int = 0
    success_protect_until: int = 0
    live_budget: int = 160
    decision: str = "EXPLORE"
    reason: str = "initial observation"
    last_signature: str | None = None
    last_core_signature: str | None = None
    seen_signatures: deque = field(default_factory=lambda: deque(maxlen=96))
    last_terms: dict = field(default_factory=dict)


class DifferenceWeightedExploitation:
    def __init__(self):
        self._states = {}
        self._lock = threading.RLock()

    def state(self, game_id):
        key = str(game_id or "unknown")
        with self._lock:
            if key not in self._states:
                self._states[key] = DWEGameState(game_id=key)
            return self._states[key]

    @staticmethod
    def _budget(weight):
        for threshold, budget in DWE_BUDGET_TIERS:
            if weight >= threshold:
                return int(min(ADVISORY_MAX_MOVES, budget))
        return int(ADVISORY_MAX_MOVES)

    def pre(self, game_id, action):
        with self._lock:
            st = self.state(game_id)
            print(
                "DWE PRE "
                f"game={st.game_id} move={st.move + 1:03d} action={action} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} mode={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"protect_until={st.success_protect_until} reason={st.reason}",
                flush=True,
            )
            return {
                "move": st.move + 1,
                "score": st.last_score,
                "levels": st.last_levels,
                "signature": st.last_signature,
                "core_signature": st.last_core_signature,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "decision": st.decision,
            }

    def post(self, game_id, action, before, after):
        with self._lock:
            st = self.state(game_id)
            st.move += 1

            before_score = _num(before.get("score"), st.last_score)
            if before_score is None:
                before_score = st.last_score
            after_score = _num(after.get("score"), None)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = before_score + reward
            score_delta = after_score - before_score

            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            level_event = bool(after.get("level_completed"))
            if after_levels is None:
                after_levels = before_levels + (1 if level_event else 0)
            level_delta = max(0, int(after_levels) - int(before_levels))
            level_event = bool(level_event or level_delta > 0)

            sig = after.get("signature")
            prev_sig = st.last_signature
            seen_before = set(st.seen_signatures)
            novel = bool(sig and sig not in seen_before)
            repeated = bool(sig and (sig == prev_sig or sig in seen_before))
            board_changed = after.get("board_changed")
            if board_changed is None and sig and prev_sig:
                board_changed = sig != prev_sig
            if board_changed is None:
                board_changed = bool(score_delta != 0 or level_event or reward != 0)

            core_sig = after.get("core_signature")
            prev_core_sig = st.last_core_signature
            core_changed = None
            if core_sig and prev_core_sig:
                core_changed = core_sig != prev_core_sig

            positive_score = score_delta > 1e-9
            positive_reward = reward > 1e-9
            meaningful_progress = bool(level_event or positive_score or positive_reward)

            # NO_IMPACT = apparent frame/board activity confined to the thin outer
            # band, with no verified reward/score/level progress.
            no_impact = bool(
                board_changed
                and core_changed is False
                and not meaningful_progress
            )
            effective_board_changed = bool(board_changed and not no_impact)
            effective_novel = bool(novel and not no_impact)
            state_activity = bool(effective_board_changed or effective_novel)
            loop_signal = bool(repeated and not meaningful_progress)

            if no_impact:
                st.no_impact_streak += 1
            else:
                st.no_impact_streak = 0

            if meaningful_progress:
                st.no_progress_streak = 0
            else:
                st.no_progress_streak += 1

            if meaningful_progress or state_activity:
                st.stall_streak = 0
            else:
                st.stall_streak += 1

            if loop_signal:
                st.repeat_streak += 1
            else:
                st.repeat_streak = 0

            # Progress value is deliberately conservative: visual novelty alone is useful
            # information, but weaker than verified score/reward/level progress.
            progress_value = 0.0
            if level_event:
                progress_value += 1.00
            if positive_score:
                progress_value += min(0.65, 0.20 + abs(score_delta))
            if positive_reward:
                progress_value += min(0.40, 0.10 + abs(reward))
            if effective_board_changed:
                progress_value += 0.12
            if effective_novel:
                progress_value += 0.10
            if loop_signal:
                progress_value -= 0.30
            if no_impact:
                progress_value -= 0.25
            if not meaningful_progress and not state_activity:
                progress_value -= 0.15
            progress_value = _clip(progress_value, -1.0, 1.0)
            st.progress_velocity = _clip(0.70 * st.progress_velocity + 0.30 * progress_value, -1.0, 1.0)

            # Causal confidence is tied only to observed current-game consequences.
            causal_confidence = 0.0
            if level_event:
                causal_confidence = 1.0
            elif positive_score:
                causal_confidence = 0.85
            elif positive_reward:
                causal_confidence = 0.70
            elif effective_board_changed and effective_novel:
                causal_confidence = 0.35
            elif effective_board_changed:
                causal_confidence = 0.20

            normalized_score = _clip(after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 2.0)
            target_proximity = 1.0 if after_score >= TARGET_MIN_GAME_SCORE else _clip(
                after_score / max(TARGET_MIN_GAME_SCORE, 1e-9), 0.0, 1.0
            )
            # A stagnant near-target game should not monopolize budget, so proximity is
            # gated by recent causal progress velocity.
            target_activity_gate = 0.20 + 0.80 * max(0.0, st.progress_velocity)
            target_signal = target_proximity * target_activity_gate
            stall_ratio = _clip(st.stall_streak / max(MAX_STALL_ACTIONS, 1), 0.0, 1.0)
            no_progress_ratio = _clip(st.no_progress_streak / max(MAX_NO_PROGRESS_ACTIONS, 1), 0.0, 1.0)
            repeat_ratio = _clip(st.repeat_streak / 4.0, 0.0, 1.0)
            no_impact_ratio = _clip(
                st.no_impact_streak / max(NO_IMPACT_STREAK_FOR_STOP, 1),
                0.0,
                1.0,
            )
            terminal_loss = bool(after.get("lost") or (after.get("game_over") and not after.get("won")))

            terms = {
                "score": EXPLOIT_WEIGHTS["score"] * normalized_score,
                "level_complete": EXPLOIT_WEIGHTS["level_complete"] * (1.0 if level_event else 0.0),
                "progress_velocity": EXPLOIT_WEIGHTS["progress_velocity"] * st.progress_velocity,
                "novel_state": EXPLOIT_WEIGHTS["novel_state"] * (1.0 if effective_novel else 0.0),
                "causal_confidence": EXPLOIT_WEIGHTS["causal_confidence"] * causal_confidence,
                "target_proximity": EXPLOIT_WEIGHTS["target_proximity"] * target_signal,
                "stall": EXPLOIT_WEIGHTS["stall"] * stall_ratio,
                "repeat_loop": EXPLOIT_WEIGHTS["repeat_loop"] * repeat_ratio,
                "no_progress": EXPLOIT_WEIGHTS["no_progress"] * no_progress_ratio,
                "no_impact": EXPLOIT_WEIGHTS["no_impact"] * no_impact_ratio,
                "terminal_loss": EXPLOIT_WEIGHTS["terminal_loss"] * (1.0 if terminal_loss else 0.0),
            }

            game_signal = sum(terms.values())
            # Strategy weight emphasizes immediate causal evidence and punishes local
            # failure more strongly than game weight, allowing CHANGE_POLICY on good games.
            strategy_signal = (
                5.00 * (1.0 if level_event else 0.0)
                + 2.75 * st.progress_velocity
                + 2.00 * causal_confidence
                + 0.50 * (1.0 if effective_novel else 0.0)
                - 2.50 * stall_ratio
                - 4.00 * repeat_ratio
                - 3.25 * no_progress_ratio
                - 4.25 * no_impact_ratio
                - 4.50 * (1.0 if terminal_loss else 0.0)
            )

            st.game_weight = _clip(
                GAME_WEIGHT_DECAY * st.game_weight + game_signal,
                -GAME_WEIGHT_LIMIT,
                GAME_WEIGHT_LIMIT,
            )
            st.strategy_weight = _clip(
                STRATEGY_WEIGHT_DECAY * st.strategy_weight + strategy_signal,
                -STRATEGY_WEIGHT_LIMIT,
                STRATEGY_WEIGHT_LIMIT,
            )
            st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight

            if level_event or positive_score or positive_reward:
                st.success_protect_until = max(
                    st.success_protect_until,
                    st.move + SUCCESS_PROTECT_ACTIONS,
                )

            st.live_budget = self._budget(st.combined_weight)
            if st.move <= st.success_protect_until:
                st.live_budget = max(st.live_budget, min(ADVISORY_MAX_MOVES, st.success_protect_until + 12))

            protected = st.move <= st.success_protect_until
            if after.get("game_over"):
                st.decision = "TERMINAL"
                st.reason = "environment reported terminal state"
            elif st.no_progress_streak >= STALL_ESCAPE_WINDOW:
                # GhostBridge outranks the observation warmup: six committed actions
                # without verified score/reward/level progress are enough to falsify
                # the current strategy class. At twelve, broaden the hypothesis space.
                st.decision = "CHANGE_POLICY"
                if st.no_progress_streak >= STALL_HARD_WINDOW:
                    st.reason = (
                        "GhostBridge hard stall: 12 no-progress actions; "
                        "reject exhausted-equivalent probes and broaden hypothesis"
                    )
                else:
                    st.reason = (
                        "GhostBridge early stall: 6 no-progress actions; "
                        "force strategy-class change"
                    )
            elif st.move < MIN_OBSERVATION_ACTIONS:
                st.decision = "EXPLORE"
                st.reason = "minimum observation window"
            elif level_event and st.combined_weight >= 3.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "verified level completion"
            elif meaningful_progress and st.combined_weight >= 1.0:
                st.decision = "EXPLOIT"
                st.reason = "verified current-game progress"
            elif protected:
                st.decision = "EXPLOIT_PROTECTED"
                st.reason = "recent success protected exploit window"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_STOP and st.game_weight < 1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact with low game value; change strategy and continue"
            elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
                st.decision = "CHANGE_POLICY"
                st.reason = "repeated no-impact actions; abandon current local strategy"
            elif st.game_weight >= 1.0 and st.strategy_weight <= -1.0:
                st.decision = "CHANGE_POLICY"
                st.reason = "game weight high while strategy weight is low"
            elif st.combined_weight >= 6.0:
                st.decision = "HARD_EXPLOIT"
                st.reason = "very high combined exploit weight"
            elif st.combined_weight >= 3.0:
                st.decision = "EXPLOIT"
                st.reason = "high combined exploit weight"
            elif st.combined_weight >= 1.0:
                st.decision = "CAUTIOUS_EXPLOIT"
                st.reason = "positive combined exploit weight"
            elif st.combined_weight >= -1.0:
                st.decision = "BALANCED"
                st.reason = "mixed current-game evidence"
            else:
                st.decision = "EXPLORE"
                st.reason = "low exploit confidence; seek information"

            st.last_score = float(after_score)
            st.last_levels = int(after_levels)
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            if core_sig:
                st.last_core_signature = core_sig
            st.last_terms = {k: round(v, 6) for k, v in terms.items()}

            event = {
                "game_id": st.game_id,
                "move": st.move,
                "action": action,
                "before_score": before_score,
                "after_score": after_score,
                "score_delta": score_delta,
                "before_levels": before_levels,
                "after_levels": after_levels,
                "level_delta": level_delta,
                "reward": reward,
                "board_changed": bool(board_changed),
                "effective_board_changed": bool(effective_board_changed),
                "core_changed": core_changed,
                "no_impact": bool(no_impact),
                "novel_state": novel,
                "effective_novel_state": effective_novel,
                "loop_signal": loop_signal,
                "progress_value": progress_value,
                "progress_velocity": st.progress_velocity,
                "causal_confidence": causal_confidence,
                "target_proximity": target_proximity,
                "stall_streak": st.stall_streak,
                "no_progress_streak": st.no_progress_streak,
                "repeat_streak": st.repeat_streak,
                "no_impact_streak": st.no_impact_streak,
                "terms": st.last_terms,
                "game_weight": st.game_weight,
                "strategy_weight": st.strategy_weight,
                "combined_weight": st.combined_weight,
                "decision": st.decision,
                "reason": st.reason,
                "live_budget": st.live_budget,
                "hard_budget": _hard_action_cap(st.game_id),
                "action_cap_policy": "all games uncapped; environment terminal state authoritative",
                "live_budget_binding": False,
                "success_protect_until": st.success_protect_until,
                "game_over": bool(after.get("game_over")),
                "won": bool(after.get("won")),
                "lost": bool(after.get("lost")),
            }
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")

            term_text = ",".join(f"{k}={v:+.2f}" for k, v in st.last_terms.items())
            print(
                "DWE POST "
                f"game={st.game_id} move={st.move:03d} action={action} "
                f"score={before_score:.6f}->{after_score:.6f} dscore={score_delta:+.6f} "
                f"levels={before_levels}->{after_levels} dlevel={level_delta:+d} reward={reward:+.4f} "
                f"changed={int(bool(board_changed))} effective_changed={int(bool(effective_board_changed))} "
                f"NO_IMPACT={int(bool(no_impact))} novel={int(novel)} loop={int(loop_signal)} "
                f"progress={progress_value:+.3f} velocity={st.progress_velocity:+.3f} "
                f"target={target_proximity:.3f} stall={st.stall_streak}/{MAX_STALL_ACTIONS} "
                f"no_progress={st.no_progress_streak}/{MAX_NO_PROGRESS_ACTIONS} "
                f"no_impact={st.no_impact_streak}/{NO_IMPACT_STREAK_FOR_STOP} "
                f"gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
                f"combined={st.combined_weight:+.3f} next={st.decision} "
                f"advisory_budget={st.live_budget} hard_cap={_hard_cap_label(st.game_id)} "
                f"protect_until={st.success_protect_until} "
                f"terms=[{term_text}] reason={st.reason}",
                flush=True,
            )
            return event

    def should_stop(self, game_id):
        """DWE is advisory; only the environment or wall clock may stop a game."""
        with self._lock:
            st = self.state(game_id)
            cap = _hard_action_cap(st.game_id)
            if cap is not None and st.move >= cap:
                return True, f"environment-reported hard action ceiling {cap}"
            return False, "DWE action-uncapped"

    def summaries(self):
        with self._lock:
            return [
                {
                    "game_id": st.game_id,
                    "moves": st.move,
                    "score": st.last_score,
                    "levels": st.last_levels,
                    "game_weight": st.game_weight,
                    "strategy_weight": st.strategy_weight,
                    "combined_weight": st.combined_weight,
                    "decision": st.decision,
                    "reason": st.reason,
                    "live_budget": st.live_budget,
                    "stall_streak": st.stall_streak,
                    "no_progress_streak": st.no_progress_streak,
                    "repeat_streak": st.repeat_streak,
                    "no_impact_streak": st.no_impact_streak,
                    "hard_action_cap": _hard_action_cap(st.game_id),
                    "live_budget_binding": False,
                }
                for st in self._states.values()
            ]


DWE_ALLOCATOR = DifferenceWeightedExploitation()
_ADL_LOGGED_MOVE_KEYS = set()
_ADL_DEBT_LOCK = threading.RLock()
_GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS = set()
_GHOSTBRIDGE_PREMOVE_LOCK = threading.RLock()


def _ghostbridge_premove_key(game_id):
    raw = str(game_id or "unknown")
    return raw, _dwe_game_key(raw)


def _ghostbridge_premove_prepare_keys(game_id, move):
    full, short = _ghostbridge_premove_key(game_id)
    with _GHOSTBRIDGE_PREMOVE_LOCK:
        _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS.add((full, int(move)))
        _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS.add((short, int(move)))


def _ghostbridge_premove_is_prepared(game_id, move):
    full, short = _ghostbridge_premove_key(game_id)
    with _GHOSTBRIDGE_PREMOVE_LOCK:
        return (full, int(move)) in _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS or (short, int(move)) in _GHOSTBRIDGE_PREMOVE_PREPARED_MOVE_KEYS


def _ghostbridge_premove_assert_prepared(game_id, move):
    if not (GHOSTBRIDGE_PREMOVE_ENABLED and GHOSTBRIDGE_PREMOVE_FAIL_CLOSED):
        return
    if not _ghostbridge_premove_is_prepared(game_id, move):
        raise RuntimeError(
            f"GHOSTBRIDGE PRE-MOVE PLAN MISSING game={game_id} move={move}; refusing real action"
        )


def _ghostbridge_premove_frame_signature(current_frame):
    try:
        return _core_signature(current_frame) or _stable_signature(current_frame)
    except Exception:
        try:
            return hashlib.sha256(repr(current_frame).encode("utf-8", errors="replace")).hexdigest()
        except Exception:
            return "unknown"


def _ghostbridge_premove_brief(*, game_id, action_num, valid_actions, current_frame, previous_step_summary):
    """Deterministic current-game pre-move director consumed by the ADL model.

    GhostBridgePreMove does not choose an action. It identifies negative space and
    writes the constraints/questions that ADL must use when producing A/B.
    """
    st = DWE_ALLOCATOR.state(game_id)
    move = int(st.move) + 1
    v5_phase = _GBV5_RUNTIME.planner.phase(_GBV5_RUNTIME.game_state.get(str(game_id), {})).value
    legal = [str(a) for a in (valid_actions or [])]

    known_causal = "none confirmed yet"
    if st.move > 0:
        if st.move <= st.success_protect_until and st.progress_velocity > 0:
            known_causal = "recent action pattern produced verified progress; preserve only its causal features"
        elif st.progress_velocity > 0.15:
            known_causal = "recent transitions show positive current-game progress velocity"
        elif st.last_score > 0 or st.last_levels > 0:
            known_causal = "current game has verified score/level progress but the immediate causal route is not fully stable"

    if st.no_progress_streak >= STALL_HARD_WINDOW:
        missing = "current hypothesis class is missing an interaction/mechanic; broaden control-object relation"
        falsified = "all exhausted-equivalent probes in the current strategy class"
        counterfactual = "a genuinely different mechanic hypothesis should alter core state, reward, score, or level trajectory"
        info_target = "which untested interaction class can change the core game state"
        guidance = "force a new hypothesis class; do not generate A/B as cosmetic variants of the stalled policy"
    elif st.no_progress_streak >= STALL_ESCAPE_WINDOW:
        missing = "strategy-class bridge: current local policy cannot produce verified progress"
        falsified = "the current strategy class after six no-progress actions"
        counterfactual = "a different strategy class should produce novel core-state evidence within a small number of actions"
        info_target = "highest-information legal test from a different strategy class"
        guidance = "make at least one candidate belong to a different strategy class"
    elif st.no_impact_streak >= NO_IMPACT_STREAK_FOR_POLICY_CHANGE:
        missing = "effective control-to-core-state connection"
        falsified = "repeated actions that only change HUD/move-band or otherwise have no causal impact"
        counterfactual = "an effective control should change a game object, mechanic, reward, score, or level state"
        info_target = "which legal control reaches a core object/mechanic rather than presentation state"
        guidance = "exclude equivalent no-impact repeats from both candidates"
    elif st.repeat_streak >= 2:
        missing = "novel transition path out of a repeated state-action basin"
        falsified = "locally repeated state/action combinations"
        counterfactual = "a useful alternative should leave the repeated state basin"
        info_target = "lowest-cost legal action with maximum transition novelty"
        guidance = "prefer candidates that are not equivalent to the repeated state-action pair"
    elif st.move <= st.success_protect_until and st.move > 0:
        missing = "minimal continuation bridge from verified success toward deeper completion"
        falsified = "unrelated exploration that abandons a recent causal success without evidence"
        counterfactual = "preserving the successful causal feature should continue score/level progress"
        info_target = "whether the successful causal feature generalizes to the next required transition"
        guidance = "bias A toward the shortest continuation of verified success; keep B as a bounded falsification probe"
    else:
        missing = "unknown current-game capability or interaction rule"
        falsified = "none yet"
        counterfactual = "a useful probe should create measurable core-state information or verified progress"
        info_target = "highest-value unresolved control/object/mechanic relation"
        guidance = "use A for the best supported causal move and B for the cleanest information-gain probe"

    constraints = [
        "one real trajectory only",
        "solver-observed evidence only",
        f"mode={st.decision}",
        f"hard_cap={_hard_cap_label(game_id)}",
    ]
    if legal:
        constraints.append("legal_actions=" + ",".join(legal))
    if st.no_progress_streak:
        constraints.append(f"no_progress_streak={st.no_progress_streak}")
    if st.no_impact_streak:
        constraints.append(f"no_impact_streak={st.no_impact_streak}")

    frame_sig = _ghostbridge_premove_frame_signature(current_frame)[:16]
    priority_context = PERFORMANCE_STORE.context(game_id)
    st.live_budget = int(priority_context["advisory_budget"])
    trajectory_prior = PERFORMANCE_STORE.trajectory_prior(game_id, frame_sig, st.move)
    failure_memory = PERFORMANCE_STORE.failure_memory(game_id)
    if priority_context["tier"] in ("PROVEN_HIGH", "PROVEN"):
        info_target = "reward-path uncertainty nearest score-producing transitions"
    elif priority_context["tier"] == "PERSISTENT_ZERO":
        info_target = "cheap recovery probe excluding remembered dead ends"
    payload = {
        "schema": "adl.arc3.ghostbridge_premove.pre_move.v1",
        "game_id": str(game_id),
        "game_key": _dwe_game_key(game_id),
        "move": move,
        "analyzer_action_num": int(action_num),
        "frame_signature": frame_sig,
        "mode": st.decision,
        "game_weight": round(float(st.game_weight), 6),
        "strategy_weight": round(float(st.strategy_weight), 6),
        "progress_velocity": round(float(st.progress_velocity), 6),
        "stall_streak": int(st.stall_streak),
        "no_progress_streak": int(st.no_progress_streak),
        "repeat_streak": int(st.repeat_streak),
        "no_impact_streak": int(st.no_impact_streak),
        "known_causal": known_causal,
        "missing_capability": missing,
        "falsified": falsified,
        "counterfactual": counterfactual,
        "info_target": info_target,
        "constraint": "; ".join(constraints),
        "adl_guidance": guidance,
        "priority_tier": priority_context["tier"],
        "priority_mode": priority_context["mode"],
        "priority_value": priority_context["priority"],
        "advisory_budget": priority_context["advisory_budget"],
        "hard_cap": None,
        "marginal_value": priority_context["marginal_value"],
        "successful_prefix_confidence": priority_context["successful_prefix_confidence"],
        "successful_prefix_length": priority_context["successful_prefix_length"],
        "trajectory_prior": trajectory_prior,
        "failure_memory": asdict(failure_memory),
    }

    # Deduplicate retries for the same pending real move while guaranteeing at
    # least one durable pre-move record before the environment action boundary.
    should_write = not _ghostbridge_premove_is_prepared(game_id, move)
    if should_write:
        with _GHOSTBRIDGE_PREMOVE_LOCK:
            if not _ghostbridge_premove_is_prepared(game_id, move):
                with GHOSTBRIDGE_PRE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                    fh.write(json.dumps(payload, sort_keys=True, default=str) + "\n")
                _ghostbridge_premove_prepare_keys(game_id, move)

    if should_write:
        print(
            "GHOSTBRIDGE PRE "
            f"game={game_id} move={move:03d} mode={st.decision} "
            f"tier={priority_context['tier']} priority={priority_context['priority']:.3f} "
            f"mode={priority_context['mode']} advisory_budget={priority_context['advisory_budget']} "
            f"hard_cap=UNCAPPED missing={missing} info_target={info_target}",
            flush=True,
        )

    return (
        "GHOSTBRIDGE_PRE_MOVE_CONTEXT\n"
        f"STEP={move}\n"
        f"GHOSTBRIDGE_V5_PHASE={v5_phase}\n"
        f"KNOWN_CAUSAL={known_causal}\n"
        f"MISSING_CAPABILITY={missing}\n"
        f"FALSIFIED={falsified}\n"
        f"COUNTERFACTUAL={counterfactual}\n"
        f"INFO_TARGET={info_target}\n"
        f"CONSTRAINT={'; '.join(constraints)}\n"
        f"ADL_GUIDANCE={guidance}\n"
        "REQUIRED_ORDER: emit GHOSTBRIDGE_PRE_MOVE_PLAN, then ADL/DWE_PRE_DECISION, then exactly one real tool action."
    )


def _ghostbridge_mark_logged(game_id, move):
    with _ADL_DEBT_LOCK:
        _ADL_LOGGED_MOVE_KEYS.add((str(game_id or "unknown"), int(move)))


def _ghostbridge_assert_no_adl_debt(game_id):
    """Fail closed: never permit move N+1 when move N lacks post-move ADL."""
    if not ADL_DEBT_RECOVERY_ENABLED:
        return
    st = DWE_ALLOCATOR.state(game_id)
    if st.move <= 0:
        return
    key = (str(st.game_id), int(st.move))
    with _ADL_DEBT_LOCK:
        if key not in _ADL_LOGGED_MOVE_KEYS:
            raise RuntimeError(
                f"UNREPAIRED ADL DEBT game={st.game_id} move={st.move}; refusing next action"
            )


def _ghostbridge_log_has_move(game_id, move):
    if not DWE_MOVE_LOG.exists():
        return False
    target_game = str(game_id or "unknown")
    target_move = int(move)
    try:
        lines = DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    except Exception:
        return False
    for raw in reversed(lines[-512:]):
        try:
            item = json.loads(raw)
        except Exception:
            continue
        if str(item.get("game_id")) == target_game and int(item.get("move", -1)) == target_move:
            return True
    return False


def _ghostbridge_post_move_adl(game_id, action, before, after):
    """Full DWE post-update first; deterministic degraded recovery on analysis/log failure."""
    st_before = DWE_ALLOCATOR.state(game_id)
    expected_move = int(st_before.move) + 1
    try:
        event = DWE_ALLOCATOR.post(game_id, action, before, after)
        _ghostbridge_mark_logged(game_id, event.get("move", expected_move))
        return event
    except Exception as exc:
        # If the full event was already persisted and only a later print failed, do not duplicate it.
        if _ghostbridge_log_has_move(game_id, expected_move):
            _ghostbridge_mark_logged(game_id, expected_move)
            print(
                f"GHOSTBRIDGE ADL DEBT REPAIRED game={game_id} move={expected_move} "
                f"mode=existing-full-record error={type(exc).__name__}:{exc}",
                flush=True,
            )
            return {"game_id": str(game_id), "move": expected_move, "recovered_existing": True}

        # Degraded deterministic record: preserve the causal transition even when rich scoring failed.
        with DWE_ALLOCATOR._lock:
            st = DWE_ALLOCATOR.state(game_id)
            if st.move < expected_move:
                st.move = expected_move
            before_score = _num(before.get("score"), st.last_score)
            after_score = _num(after.get("score"), before_score)
            reward = _num(after.get("reward"), 0.0) or 0.0
            if after_score is None:
                after_score = (before_score or 0.0) + reward
            before_levels = before.get("levels")
            if before_levels is None:
                before_levels = st.last_levels
            after_levels = after.get("levels")
            if after_levels is None:
                after_levels = before_levels + (1 if after.get("level_completed") else 0)
            sig = after.get("signature")
            if sig:
                st.last_signature = sig
                st.seen_signatures.append(sig)
            core_sig = after.get("core_signature")
            if core_sig:
                st.last_core_signature = core_sig
            st.last_score = float(after_score or 0.0)
            st.last_levels = int(after_levels or 0)
            st.decision = "CHANGE_POLICY"
            st.reason = "GhostBridge degraded post-move ADL recovery"

        fallback = {
            "game_id": str(game_id or "unknown"),
            "move": expected_move,
            "action": action,
            "before_score": before_score,
            "after_score": after_score,
            "score_delta": (float(after_score or 0.0) - float(before_score or 0.0)),
            "before_levels": before_levels,
            "after_levels": after_levels,
            "level_delta": max(0, int(after_levels or 0) - int(before_levels or 0)),
            "reward": reward,
            "board_changed": after.get("board_changed"),
            "novel_state": None,
            "loop_signal": None,
            "progress_value": None,
            "decision": "CHANGE_POLICY",
            "reason": "GhostBridge degraded post-move ADL recovery",
            "hard_budget": _hard_action_cap(str(game_id)),
            "action_cap_policy": "all games uncapped; environment terminal state authoritative",
            "adl_debt_recovered": True,
            "recovery_error": f"{type(exc).__name__}: {exc}",
            "game_over": bool(after.get("game_over")),
            "won": bool(after.get("won")),
            "lost": bool(after.get("lost")),
        }
        try:
            with DWE_MOVE_LOG.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(fallback, sort_keys=True, default=str) + "\n")
        except Exception as log_exc:
            raise RuntimeError(
                f"POST-MOVE ADL FAILED AND RECOVERY COULD NOT BE PERSISTED "
                f"game={game_id} move={expected_move}: {log_exc}"
            ) from log_exc
        _ghostbridge_mark_logged(game_id, expected_move)
        print(
            f"GHOSTBRIDGE ADL DEBT REPAIRED game={game_id} move={expected_move} "
            f"mode=degraded-transition-record error={type(exc).__name__}:{exc}",
            flush=True,
        )
        return fallback


_DWE_ACTION_DEPTH = contextvars.ContextVar("dwe_action_depth", default=0)
_DWE_PATCHED_ACTION_METHODS = []
_DWE_PATCHED_STOP_METHODS = []


def _method_action_score(name, method):
    score = 0
    lname = name.lower()
    if lname in {"step", "act", "action", "execute_action", "perform_action", "take_action", "play_action", "apply_action"}:
        score += 8
    if "action" in lname:
        score += 4
    if any(token in lname for token in ("step", "move", "act", "play")):
        score += 2
    try:
        sig = inspect.signature(method)
        params = {p.lower() for p in sig.parameters}
        if params.intersection({"action", "action_name", "action_spec", "move", "command"}):
            score += 8
    except Exception:
        pass
    try:
        source = inspect.getsource(method).lower()
        if "board_changed" in source or "level_completed" in source or ".step(" in source:
            score += 4
    except Exception:
        pass
    return score


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            _ghostbridge_post_move_adl(game_id, action, before, after)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self)
            action = _extract_action(args, kwargs)
            before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            after = _snapshot(self, result)
            _ghostbridge_post_move_adl(game_id, action, before, after)
            return result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_gameapi_action_hook(game_apis):
    classes = []
    for api in game_apis:
        if api.__class__ not in classes:
            classes.append(api.__class__)
    installed = []
    for cls in classes:
        candidates = []
        for name in dir(cls):
            if name.startswith("_"):
                continue
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if not callable(method):
                continue
            score = _method_action_score(name, method)
            if score > 0:
                candidates.append((score, name))
        candidates.sort(reverse=True)
        if not candidates:
            raise RuntimeError(
                f"DWE could not identify a real action method on {cls.__module__}.{cls.__name__}; "
                "refusing to run without per-move exploit logging."
            )
        best_score, best_name = candidates[0]
        if best_score < 6:
            raise RuntimeError(
                f"DWE action-boundary confidence too low for {cls.__name__}: {candidates[:8]}"
            )
        if _wrap_action_method(cls, best_name):
            installed.append(f"{cls.__module__}.{cls.__name__}.{best_name}")
        print(
            f"DWE ACTION HOOK class={cls.__module__}.{cls.__name__} "
            f"method={best_name} confidence={best_score} candidates={candidates[:6]}",
            flush=True,
        )
    return installed


def _object_game_id(self, args, kwargs):
    return _extract_game_id(self, *args, *kwargs.values())


def _wrap_should_stop(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_stop_wrapper", False):
        return False

    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            original_result = await original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result
    else:
        def wrapped(self, *args, **kwargs):
            original_result = original(self, *args, **kwargs)
            if original_result:
                return original_result
            gid = _object_game_id(self, args, kwargs)
            stop, reason = DWE_ALLOCATOR.should_stop(gid)
            if stop:
                print(f"DWE STOP game={gid} reason={reason}", flush=True)
                return True
            return original_result

    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_stop_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_STOP_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


def _install_should_stop_hooks(game_apis):
    classes = {api.__class__ for api in game_apis}
    # The stop predicate can live on GameAPI, a solver/session class, or a run class.
    for module_name, module in list(sys.modules.items()):
        if not module or not (
            module_name.startswith("taaf") or module_name.startswith("inference")
        ):
            continue
        try:
            values = list(vars(module).values())
        except Exception:
            continue
        for obj in values:
            if inspect.isclass(obj):
                classes.add(obj)

    installed = []
    for cls in classes:
        for name in ("should_stop", "_should_stop"):
            try:
                method = getattr(cls, name)
            except Exception:
                continue
            if callable(method) and _wrap_should_stop(cls, name):
                full = f"{cls.__module__}.{cls.__name__}.{name}"
                installed.append(full)
                print(f"DWE STOP HOOK {full}", flush=True)
    if not installed:
        print(
            "DWE STOP HOOK WARNING: no should_stop predicate found; hard ceiling remains active. "
            "Per-move DWE logging and policy weighting are still active.",
            flush=True,
        )
    return installed


def _install_dwe_runtime_hooks(game_apis):
    action_hooks = _install_gameapi_action_hook(game_apis)
    stop_hooks = _install_should_stop_hooks(game_apis)
    if not action_hooks and not _DWE_PATCHED_ACTION_METHODS:
        raise RuntimeError("DWE requires an action-boundary hook; none was installed.")
    print(
        "DWE RUNTIME ACTIVE "
        f"action_hooks={len(action_hooks) or len(_DWE_PATCHED_ACTION_METHODS)} "
        f"stop_hooks={len(stop_hooks) or len(_DWE_PATCHED_STOP_METHODS)} "
        "log_every_move=True",
        flush=True,
    )


print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("DWE ACTIVE: separate game and strategy exploit weights", flush=True)
print("GHOSTBRIDGE ACTIVE: negative-space learning + fail-closed ADL debt recovery", flush=True)
print("DWE LOGGING: PRE + POST for every real environment action", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME SCORE MEMORY: ENABLED (structured score evidence and score-validated rule utility only)", flush=True)

print("ACTION CAP POLICY: all games action-UNCAPPED; environment terminal state authoritative", flush=True)


In [ ]:
\
# === DWE v3 OVERLAY: HUD/NO-IMPACT + RESULT FEEDBACK ===
import inspect
import json
import types
from collections import deque
from typing import Mapping
DWE_V3_MOVE_LOG = WORKING_DIR / "dwe_v3_move_events.jsonl"
DWE_V3_MOVE_LOG.unlink(missing_ok=True)
_DWE_V3_TRACKERS = {}


def _dwe_grid(value, seen=None):
    if value is None:
        return None
    if seen is None:
        seen = set()
    try:
        ident = id(value)
        if ident in seen:
            return None
        seen.add(ident)
    except Exception:
        pass
    try:
        if hasattr(value, "tolist"):
            value = value.tolist()
    except Exception:
        pass
    if isinstance(value, Mapping):
        for key in ("grid","board","frame","pixels","ascii","data","array","state_matrix","current_frame","after_frame"):
            if key in value:
                got = _dwe_grid(value[key], seen)
                if got is not None:
                    return got
        return None
    if isinstance(value, str):
        lines = [line.rstrip() for line in value.splitlines() if line.strip()]
        rows = []
        for line in lines:
            parts = line.split()
            row = parts if len(parts) > 1 else list(line)
            if row:
                rows.append(tuple(str(x) for x in row))
        if len(rows) >= 2 and len({len(row) for row in rows}) == 1 and len(rows[0]) >= 2:
            return tuple(rows)
        return None
    if isinstance(value, (list, tuple)) and value and all(isinstance(row, (list, tuple)) for row in value):
        widths = {len(row) for row in value}
        if len(widths) == 1 and next(iter(widths), 0) > 0:
            return tuple(tuple(str(x) for x in row) for row in value)
    for attr in ("grid","board","ascii","pixels","array","data","frame","current_frame","after_frame"):
        try:
            child = getattr(value, attr)
        except Exception:
            continue
        if callable(child):
            continue
        got = _dwe_grid(child, seen)
        if got is not None:
            return got
    return None


def _dwe_extract_grid(*objs):
    for obj in objs:
        for candidate in _walk_candidates(obj, max_depth=3):
            grid = _dwe_grid(candidate)
            if grid is not None:
                return grid
    return None


def _dwe_diff(a, b):
    if a is None or b is None or len(a) != len(b):
        return None
    if any(len(x) != len(y) for x, y in zip(a, b)):
        return None
    return {(r,c) for r,(ra,rb) in enumerate(zip(a,b)) for c,(x,y) in enumerate(zip(ra,rb)) if x != y}


def _dwe_masked_signature(grid, rows=(), cols=()):
    if grid is None:
        return None
    rows, cols = set(rows), set(cols)
    payload = [["." if r in rows or c in cols else str(v) for c,v in enumerate(row)] for r,row in enumerate(grid)]
    raw = json.dumps(payload, separators=(",",":"), ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8", "replace")).hexdigest()[:16]


def _dwe_tracker(game_id):
    key = str(game_id or "unknown")
    if key not in _DWE_V3_TRACKERS:
        _DWE_V3_TRACKERS[key] = {
            "history": deque(maxlen=NO_IMPACT_BAND_WINDOW),
            "shape": None,
            "band_rows": (),
            "band_cols": (),
            "no_impact_streak": 0,
            "no_impact_total": 0,
            "last_source": "none",
        }
    return _DWE_V3_TRACKERS[key]


def _dwe_band(t):
    history = list(t["history"])
    if len(history) < NO_IMPACT_BAND_WARMUP:
        return (), ()
    denom = float(len(history))
    rc, cc = {}, {}
    for rows, cols in history:
        for r in rows:
            rc[r] = rc.get(r, 0) + 1
        for c in cols:
            cc[c] = cc.get(c, 0) + 1
    return (
        tuple(sorted(r for r,n in rc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
        tuple(sorted(c for c,n in cc.items() if n/denom >= NO_IMPACT_BAND_THRESHOLD)),
    )


def _dwe_classify(game_id, before_grid, after_grid, meaningful_progress):
    t = _dwe_tracker(game_id)
    changed = _dwe_diff(before_grid, after_grid)
    if changed is None:
        t["no_impact_streak"] = 0
        t["last_source"] = "unavailable"
        return False, "unavailable", t["band_rows"], t["band_cols"], None
    shape = (len(after_grid or ()), len(after_grid[0]) if after_grid else 0)
    if t["shape"] is not None and shape != t["shape"]:
        t["history"].clear(); t["band_rows"] = (); t["band_cols"] = (); t["no_impact_streak"] = 0
    t["shape"] = shape
    prior_rows, prior_cols = _dwe_band(t)
    no_impact = bool(changed and not meaningful_progress and (prior_rows or prior_cols) and all(r in prior_rows or c in prior_cols for r,c in changed))
    t["history"].append((tuple(sorted({r for r,_ in changed})), tuple(sorted({c for _,c in changed}))))
    t["band_rows"], t["band_cols"] = _dwe_band(t)
    if no_impact:
        t["no_impact_streak"] += 1; t["no_impact_total"] += 1; t["last_source"] = "band"
    else:
        t["no_impact_streak"] = 0; t["last_source"] = "exact-static" if not changed else "band-learning"
    canonical = _dwe_masked_signature(after_grid, t["band_rows"], t["band_cols"])
    return no_impact, t["last_source"], t["band_rows"], t["band_cols"], canonical


_DWE_V2_SNAPSHOT = _snapshot

def _snapshot(api, result=None):
    snap = _DWE_V2_SNAPSHOT(api, result)
    snap["grid"] = _dwe_extract_grid(result, api)
    return snap


_DWE_V2_PRE = DWE_ALLOCATOR.pre
_DWE_V2_POST = DWE_ALLOCATOR.post
_DWE_V2_SUMMARIES = DWE_ALLOCATOR.summaries


def _dwe_v3_pre(self, game_id, action):
    out = _DWE_V2_PRE(game_id, action)
    t = _dwe_tracker(game_id)
    print(
        "DWE PRE+ "
        f"game={game_id} action={action} no_impact={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} "
        f"hud_rows={list(t['band_rows'])} hud_cols={list(t['band_cols'])} seed={CONTROL_SEED}",
        flush=True,
    )
    return out


def _dwe_v3_post(self, game_id, action, before, after):
    bscore = _num(before.get("score"), 0.0) or 0.0
    ascore = _num(after.get("score"), bscore)
    reward = _num(after.get("reward"), 0.0) or 0.0
    level_event = bool(after.get("level_completed"))
    meaningful = bool(level_event or (ascore is not None and ascore > bscore + 1e-9) or reward > 1e-9)
    no_impact, source, rows, cols, canonical = _dwe_classify(game_id, before.get("grid"), after.get("grid"), meaningful)
    adjusted = dict(after)
    if canonical:
        adjusted["signature"] = canonical
    if no_impact:
        adjusted["board_changed"] = False
    event = _DWE_V2_POST(game_id, action, before, adjusted)
    st = self.state(game_id)
    t = _dwe_tracker(game_id)
    ratio = _clip(t["no_impact_streak"] / max(MAX_NO_IMPACT_ACTIONS, 1), 0.0, 1.0)
    no_impact_term = EXPLOIT_WEIGHTS["no_impact"] * ratio
    if no_impact_term:
        st.game_weight = _clip(st.game_weight + 0.20 * no_impact_term, -GAME_WEIGHT_LIMIT, GAME_WEIGHT_LIMIT)
        st.strategy_weight = _clip(st.strategy_weight + no_impact_term, -STRATEGY_WEIGHT_LIMIT, STRATEGY_WEIGHT_LIMIT)
        st.combined_weight = 0.55 * st.game_weight + 0.45 * st.strategy_weight
        st.live_budget = self._budget(st.combined_weight)
    protected = st.move <= st.success_protect_until
    if not protected and t["no_impact_streak"] >= MAX_NO_IMPACT_ACTIONS:
        if st.game_weight >= 0.5:
            st.decision = "CHANGE_POLICY"; st.reason = "repeated housekeeping-only/no-impact actions"
        elif st.no_progress_streak >= MAX_STALL_ACTIONS:
            st.decision = "CHANGE_POLICY"; st.reason = "no-impact streak plus low game value"
        else:
            st.decision = "CHANGE_POLICY"; st.reason = "no-impact threshold reached"
    event.update({
        "no_impact": bool(no_impact), "no_impact_source": source,
        "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
        "hud_band_rows": list(rows), "hud_band_cols": list(cols), "no_impact_term": no_impact_term,
        "game_weight": st.game_weight, "strategy_weight": st.strategy_weight,
        "combined_weight": st.combined_weight, "decision": st.decision, "reason": st.reason,
        "live_budget": st.live_budget, "control_seed": CONTROL_SEED, "model_id": ANALYZER_MODEL_ID,
    })
    try:
        PERFORMANCE_STORE.record_transition(game_id, action, before, adjusted, event)
        priority_context = PERFORMANCE_STORE.context(game_id)
        st.live_budget = int(priority_context["advisory_budget"])
        st.decision = priority_context["mode"] if st.decision not in ("TERMINAL", "CHANGE_POLICY") else st.decision
        event.update({
            "priority_tier": priority_context["tier"],
            "priority_mode": priority_context["mode"],
            "priority_value": priority_context["priority"],
            "marginal_value": priority_context["marginal_value"],
            "advisory_budget": priority_context["advisory_budget"],
            "hard_cap": None,
        })
    except Exception as priority_exc:
        print(f"PRIORITY TRANSITION WARNING game={game_id} move={st.move}: {type(priority_exc).__name__}: {priority_exc}", flush=True)
        priority_context = PERFORMANCE_STORE.context(game_id)
    with DWE_V3_MOVE_LOG.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(event, sort_keys=True, default=str) + "\n")
    print(
        "DWE POST+ "
        f"game={game_id} move={st.move:03d} action={action} no_impact={int(no_impact)} source={source} "
        f"no_impact_streak={t['no_impact_streak']}/{MAX_NO_IMPACT_ACTIONS} hud_rows={list(rows)} hud_cols={list(cols)} "
        f"no_impact_term={no_impact_term:+.3f} gameW={st.game_weight:+.3f} strategyW={st.strategy_weight:+.3f} "
        f"combined={st.combined_weight:+.3f} next={st.decision} tier={priority_context['tier']} "
        f"priority={priority_context['priority']:.3f} marginal={priority_context['marginal_value']:.6f} "
        f"advisory_budget={st.live_budget} hard_cap=UNCAPPED reason={st.reason}",
        flush=True,
    )
    return event


def _dwe_v3_summaries(self):
    items = _DWE_V2_SUMMARIES()
    for item in items:
        t = _dwe_tracker(item["game_id"])
        item.update({
            "no_impact_streak": t["no_impact_streak"], "no_impact_total": t["no_impact_total"],
            "last_no_impact_source": t["last_source"], "hud_band_rows": list(t["band_rows"]), "hud_band_cols": list(t["band_cols"]),
        })
    return items


DWE_ALLOCATOR.pre = types.MethodType(_dwe_v3_pre, DWE_ALLOCATOR)
DWE_ALLOCATOR.post = types.MethodType(_dwe_v3_post, DWE_ALLOCATOR)
DWE_ALLOCATOR.summaries = types.MethodType(_dwe_v3_summaries, DWE_ALLOCATOR)


def _dwe_annotate_result(result, event):
    if result is None:
        return
    patch = {
        "dwe_decision": event.get("decision"), "dwe_game_weight": event.get("game_weight"),
        "dwe_strategy_weight": event.get("strategy_weight"), "dwe_combined_weight": event.get("combined_weight"),
        "dwe_no_impact": event.get("no_impact"), "dwe_no_impact_source": event.get("no_impact_source"),
        "dwe_live_budget": event.get("live_budget"), "dwe_reason": event.get("reason"),
    }
    if isinstance(result, dict):
        result.update(patch); return
    for key, value in patch.items():
        try:
            setattr(result, key, value)
        except Exception:
            pass


def _wrap_action_method(cls, name):
    original = getattr(cls, name)
    if getattr(original, "_dwe_action_wrapper", False):
        return False
    if inspect.iscoroutinefunction(original):
        async def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return await original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _GBV5_RUNTIME.assert_zero_debt(str(game_id))
            _gbv5_pre = _GBV5_RUNTIME.prepare(str(game_id), action, before)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = await original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            _gbv5_after = _snapshot(self, result)
            _gbv5_envelope = _GBV5_RUNTIME.commit(str(game_id), _gbv5_after)
            event = _ghostbridge_post_move_adl(game_id, action, before, _gbv5_after)
            event["ghostbridge_v5_transition_id"] = _gbv5_envelope.transition_id
            event["ghostbridge_v5_phase"] = _gbv5_envelope.planner_phase.value
            _dwe_annotate_result(result, event)
            return result
    else:
        def wrapped(self, *args, **kwargs):
            depth = _DWE_ACTION_DEPTH.get()
            if depth:
                return original(self, *args, **kwargs)
            token = _DWE_ACTION_DEPTH.set(depth + 1)
            game_id = _extract_game_id(self); action = _extract_action(args, kwargs); before = _snapshot(self)
            _ghostbridge_assert_no_adl_debt(game_id)
            _GBV5_RUNTIME.assert_zero_debt(str(game_id))
            _gbv5_pre = _GBV5_RUNTIME.prepare(str(game_id), action, before)
            _ghostbridge_premove_assert_prepared(game_id, DWE_ALLOCATOR.state(game_id).move + 1)
            DWE_ALLOCATOR.pre(game_id, action)
            try:
                result = original(self, *args, **kwargs)
            finally:
                _DWE_ACTION_DEPTH.reset(token)
            _gbv5_after = _snapshot(self, result)
            _gbv5_envelope = _GBV5_RUNTIME.commit(str(game_id), _gbv5_after)
            event = _ghostbridge_post_move_adl(game_id, action, before, _gbv5_after)
            event["ghostbridge_v5_transition_id"] = _gbv5_envelope.transition_id
            event["ghostbridge_v5_phase"] = _gbv5_envelope.planner_phase.value
            _dwe_annotate_result(result, event)
            return result
    wrapped.__name__ = getattr(original, "__name__", name)
    wrapped.__doc__ = getattr(original, "__doc__", None)
    wrapped._dwe_action_wrapper = True
    wrapped._dwe_original = original
    setattr(cls, name, wrapped)
    _DWE_PATCHED_ACTION_METHODS.append(f"{cls.__module__}.{cls.__name__}.{name}")
    return True


print(f"DWE v3 OVERLAY ACTIVE seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} no_impact=statistical-band result_feedback=on", flush=True)


## 8. Run exactly one real competition trajectory per game

The 3.57 control execution plane is preserved: one pass, no environment replay selection, no speculative forks. GhostBridge PRE-MOVE planning happens inside the analyzer prompt path and consumes **zero extra environment actions**. POST_MOVE_ADL is written after every committed action before another action is permitted.


In [ ]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
# === AUTOLOAD STAGE 7A — HARD GUARD BEFORE ANY GAME OBJECT / REAL ACTION ===
for _audit_path in (WORKING_DIR / "auto_input_manifest.json", WORKING_DIR / "auto_runtime_audit.json"):
    if not _audit_path.is_file():
        raise RuntimeError(f"AUTOLOAD AUDIT MISSING BEFORE GAMEPLAY: {_audit_path}")
_auto_inputs = json.loads((WORKING_DIR / "auto_input_manifest.json").read_text(encoding="utf-8"))
_auto_runtime = json.loads((WORKING_DIR / "auto_runtime_audit.json").read_text(encoding="utf-8"))
if _auto_inputs.get("expected_served_model") != ANALYZER_MODEL_ID:
    raise RuntimeError("AUTOLOAD input manifest lost the required Qwen3.8 model identity")
if not _auto_runtime.get("completion_smoke_pass"):
    raise RuntimeError("AUTOLOAD model completion smoke test did not pass")
if _auto_runtime.get("requirements_lock_failures"):
    raise RuntimeError("AUTOLOAD pinned runtime has unresolved requirements")
if _auto_runtime.get("recursive_project_dependency_install") is not False:
    raise RuntimeError("Unsafe recursive source dependency installer detected")
print("AUTOLOAD STAGE 7A PASS — INPUT/RUNTIME AUDITS VERIFIED; GAMEPLAY MAY START", flush=True)

import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(os.environ["ARC_AGI3_ENVIRONMENTS_DIR"])
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# Game scheduling is distinct from action selection. Rank by persisted score evidence;
# ties remain deterministic. Every tier retains a nonzero scheduler weight.
_game_ids = [_extract_game_id(api) for api in game_apis]
PERFORMANCE_STORE.print_ranking(_game_ids)
_rank_order = {gid: rank for rank, (_, _, gid, _, _) in enumerate(PERFORMANCE_STORE.ranked(_game_ids))}
game_apis.sort(key=lambda api: _rank_order.get(_extract_game_id(api), RUN_GAME_COUNT))
for _api in game_apis:
    _gid = _extract_game_id(_api)
    PERFORMANCE_STORE.begin_episode(_gid, f"run-{CONTROL_SEED}-{_gid}")
    _ctx = PERFORMANCE_STORE.context(_gid)
    print(
        f"GAME PRIORITY game={_gid} tier={_ctx['tier']} priority={_ctx['priority']:.3f} "
        f"best_score={_ctx['best_score']:.6f} mean_score={_ctx['mean_score']:.6f} "
        f"positive_rate={_ctx['positive_rate']:.3f} score_per_action={_ctx['score_per_action']:.6f} "
        f"mode={_ctx['mode']} advisory_budget={_ctx['advisory_budget']} hard_cap=UNCAPPED",
        flush=True,
    )

# Validate per-game action-cap mapping before gameplay.
_cap_contract = {_extract_game_id(api): _hard_action_cap(_extract_game_id(api)) for api in game_apis}
_unexpected_caps = [(gid, cap) for gid, cap in _cap_contract.items() if cap is not None]
if _unexpected_caps:
    raise RuntimeError(f"Notebook unexpectedly injected hard action caps: {_unexpected_caps}")
print("PIPELINE STAGE 8 — ACTION CAP CONTRACT all_games=UNCAPPED", flush=True)

# Install deterministic per-move exploit logging and stop-loss hooks only after
# the actual GameAPI class(es) have been constructed. The notebook refuses to
# start the benchmark if it cannot identify the action boundary confidently.
_install_dwe_runtime_hooks(game_apis)

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
# Wall-clock safety remains independent of action caps. Non-ls20 games are action-uncapped, not time-unlimited.
RUN_PER_GAME_SECONDS = min(
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else 7920.0,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "ls20_max_moves": LS20_MAX_MOVES,
    "action_cap_policy": {"ls20": None, "all_other_games": None},
    "global_action_limit_sentinel": GLOBAL_UNCAPPED_ACTION_LIMIT,
    "dwe_stop_loss_binding": False,
    "dwe_live_budget_binding": False,
    "control_seed": CONTROL_SEED,
    "analyzer_model": os.environ.get("INFERENCE_ANALYZER_MODEL"),
    "qwen_model_dir": str(QWEN_MODEL_DIR),
    "resolved_model_dataset": str(QWEN_MODEL_DIR),
    "context_window": int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768")),
    "frame_mode": os.environ.get("ARC3_FRAME_MODE"),
    "state_graph": os.environ.get("ARC3_STATE_GRAPH"),
    "no_impact_weight": EXPLOIT_WEIGHTS["no_impact"],
    "no_impact_policy_change": NO_IMPACT_STREAK_FOR_POLICY_CHANGE,
    "no_impact_stop": NO_IMPACT_STREAK_FOR_STOP,
    "schema": "adldb.arc3.duckv12.control357.ghostbridge_premove.v1",
    "dwe_enabled": True,
    "ghostbridge_enabled": True,
    "ghostbridge_premove_enabled": GHOSTBRIDGE_PREMOVE_ENABLED,
    "ghostbridge_premove_fail_closed": GHOSTBRIDGE_PREMOVE_FAIL_CLOSED,
    "ghostbridge_premove_order": "pre_move_brief -> ADL/DWE decision -> one real action -> post_move_ADL",
    "ghostbridge_premove_log": str(GHOSTBRIDGE_PRE_MOVE_LOG),
    "adl_debt_recovery": True,
    "stall_escape_window": STALL_ESCAPE_WINDOW,
    "stall_hard_window": STALL_HARD_WINDOW,
    "dwe_log_every_move": True,
    "dwe_weights": EXPLOIT_WEIGHTS,
    "dwe_budget_tiers": [list(x) for x in DWE_BUDGET_TIERS],
    "success_protect_actions": SUCCESS_PROTECT_ACTIONS,
    "min_observation_actions": MIN_OBSERVATION_ACTIONS,
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_external_prior": True,
    "second_environment_pass": False,
    "target_score_games": TARGET_SCORE_GAMES,
    "target_min_game_score": TARGET_MIN_GAME_SCORE,
    "max_stall_actions": MAX_STALL_ACTIONS,
    "max_no_progress_actions": MAX_NO_PROGRESS_ACTIONS,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUCK-V12 3.57 CONTROL + ADL + GHOSTBRIDGE PRE-MOVE SCORED RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f} "
    f"action_caps=all:UNCAPPED advisory_max={ADVISORY_MAX_MOVES} "
    f"dwe_stop_loss_binding=off dwe_live_budget_binding=off "
    f"dwe=on log_every_move=on "
    f"seed={CONTROL_SEED} frame=full model={os.environ.get('INFERENCE_ANALYZER_MODEL')} baseline=wellkilo/arc3lab-duck-v12-control-seed-20260819",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    _gid = _extract_game_id(run)
    _prior_context = PERFORMANCE_STORE.context(_gid)
    PERFORMANCE_STORE.finish_episode(
        _gid,
        score=_run_score(run),
        actions=_run_actions(run),
        levels=_run_levels(run),
    )
    PERFORMANCE_STORE.release_unused(
        _prior_context["tier"],
        _prior_context["advisory_budget"],
        _run_actions(run),
    )
    print(
        "GHOSTBRIDGE SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )

PERFORMANCE_STORE.print_ranking([_extract_game_id(run) for run in bm.game_runs])
PERFORMANCE_STORE.print_allocation()


## 9. Write and validate `submission.parquet`


In [ ]:
# === VALIDATE FRAMEWORK-GENERATED COMPETITION ARTIFACT ===
import hashlib
import pandas as pd

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
EXPECTED_SUBMISSION_COLUMNS = ["row_id", "game_id", "end_of_game", "score"]

# The ARC gateway/framework owns score calculation and normally writes this file.
# A local-only fallback keeps public validation reproducible, but a scored rerun
# must never overwrite the gateway's official artifact.
if not SUBMISSION_PATH.is_file():
    if TRUE_SUBMISSION:
        raise RuntimeError("Competition gateway did not generate submission.parquet")
    local_rows = [
        {
            "row_id": f"{run.game_id}_0",
            "game_id": str(run.game_id),
            "end_of_game": _won(run),
            "score": _run_score(run),
        }
        for run in bm.game_runs
    ]
    pd.DataFrame(local_rows, columns=EXPECTED_SUBMISSION_COLUMNS).to_parquet(
        SUBMISSION_PATH, index=False
    )

raw_submission = SUBMISSION_PATH.read_bytes()
if len(raw_submission) < 12 or raw_submission[:4] != b"PAR1" or raw_submission[-4:] != b"PAR1":
    raise RuntimeError("submission.parquet failed Parquet magic-byte validation")

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != EXPECTED_SUBMISSION_COLUMNS:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(f"Submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}")
if check["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs")
expected_game_ids = {_extract_game_id(api) for api in game_apis}
artifact_game_ids = set(check["game_id"].astype(str))
if artifact_game_ids != expected_game_ids:
    raise RuntimeError(
        f"Submission game coverage mismatch: missing={sorted(expected_game_ids-artifact_game_ids)} "
        f"extra={sorted(artifact_game_ids-expected_game_ids)}"
    )
if check["score"].isna().any() or check["end_of_game"].isna().any():
    raise RuntimeError("Submission contains missing score/end_of_game values")

artifact_validation = {
    "schema": "arc3.submission.validation.v2",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "framework_generated": True,
    "path": str(SUBMISSION_PATH),
    "rows": int(len(check)),
    "columns": list(check.columns),
    "unique_games": int(check["game_id"].astype(str).nunique()),
    "score_sum": float(check["score"].sum()),
    "bytes": len(raw_submission),
    "sha256": hashlib.sha256(raw_submission).hexdigest(),
    "parquet_magic_valid": True,
}
(WORKING_DIR / "submission_validation.json").write_text(
    json.dumps(artifact_validation, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(
    "SUBMISSION VALIDATED "
    f"path={SUBMISSION_PATH} rows={len(check)} games={RUN_GAME_COUNT} "
    f"score_sum={float(check['score'].sum()):.6f} "
    f"sha256={artifact_validation['sha256']}",
    flush=True,
)


## 10. Final ADLDB/DWE run summary

Summarize scored outcomes and the final exploit state of every game. The output keeps the score metrics needed for the next ADL comparison and writes a compact per-game DWE summary artifact.


In [ ]:
# === FINAL ADLDB / DWE SUMMARY ===
import statistics

runs = list(getattr(bm, "game_runs", []) or [])
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]
positive = sum(score > 0 for score in scores)
at_target = sum(score >= TARGET_MIN_GAME_SCORE for score in scores)

print(
    f"ADLDB SUMMARY model={ANALYZER_MODEL_ID} seed={CONTROL_SEED} "
    f"games={len(runs)} "
    f"mean_score={(sum(scores) / len(scores) if scores else 0.0):.6f} "
    f"score_sum={sum(scores):.6f} "
    f"positive_games={positive} "
    f"target_games={at_target}/{TARGET_SCORE_GAMES} "
    f"levels={sum(levels)} "
    f"actions={sum(actions)}",
    flush=True,
)

summaries = DWE_ALLOCATOR.summaries()
with DWE_SUMMARY_LOG.open("w", encoding="utf-8") as fh:
    for item in sorted(summaries, key=lambda x: x["game_id"]):
        fh.write(json.dumps(item, sort_keys=True) + "\n")
        print(
            "DWE GAME SUMMARY "
            f"game={item['game_id']} moves={item['moves']} score={item['score']:.6f} "
            f"levels={item['levels']} gameW={item['game_weight']:+.3f} "
            f"strategyW={item['strategy_weight']:+.3f} combined={item['combined_weight']:+.3f} "
            f"decision={item['decision']} live_budget={item['live_budget']} "
            f"stall={item['stall_streak']} no_progress={item['no_progress_streak']} "
            f"no_impact={item.get('no_impact_streak', 0)} total_no_impact={item.get('no_impact_total', 0)} "
            f"repeat={item['repeat_streak']} reason={item['reason']}",
            flush=True,
        )

print(f"DWE BASE MOVE LOG: {DWE_MOVE_LOG}", flush=True)
print(f"DWE v3 MOVE LOG: {DWE_V3_MOVE_LOG}", flush=True)
print(f"DWE SUMMARY LOG: {DWE_SUMMARY_LOG}", flush=True)


## 11. Per-move exploit/ADL audit

Verify that the deterministic DWE action-boundary logger produced one unique post-action exploit record for every recorded game action. In local validation mode, incomplete coverage is a hard failure. In official competition reruns it is surfaced prominently without destroying an otherwise valid submission artifact.


In [ ]:
# === PER-MOVE DWE / POST-MOVE ADL AUDIT ===
from collections import Counter

records = []
if DWE_MOVE_LOG.exists():
    for raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("DWE AUDIT malformed line:", raw[:240], flush=True)

unique_move_keys = {
    (str(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
recorded_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
logged_moves = len(unique_move_keys)
coverage = logged_moves / recorded_actions if recorded_actions else 1.0

modes = Counter(str(item.get("decision", "unknown")) for item in records)
stop_loss = modes.get("STOP_LOSS", 0)
no_impact_moves = sum(1 for item in records if item.get("no_impact"))
debt_recoveries = sum(1 for item in records if item.get("adl_debt_recovered"))
change_policy = modes.get("CHANGE_POLICY", 0)
exploit_moves = sum(
    count for mode, count in modes.items()
    if "EXPLOIT" in mode
)

print(
    "DWE MOVE AUDIT "
    f"recorded_actions={recorded_actions} "
    f"unique_logged_moves={logged_moves} "
    f"coverage={coverage:.3f} "
    f"exploit_updates={exploit_moves} "
    f"change_policy_updates={change_policy} "
    f"stop_loss_updates={stop_loss} "
    f"no_impact_moves={no_impact_moves} "
        f"adl_debt_recoveries={debt_recoveries} "
    f"modes={dict(sorted(modes.items()))}",
    flush=True,
)

# Per-game coverage makes any missing trace immediately visible in notebook logs.
logged_by_game = Counter(str(item.get("game_id", "unknown")) for item in records)
for run in getattr(bm, "game_runs", []) or []:
    gid = _game_key(run.game_id)
    # Match either exact full id or normalized game key.
    logged = sum(
        count for key, count in logged_by_game.items()
        if _game_key(key) == gid
    )
    expected = _run_actions(run)
    game_cov = logged / expected if expected else 1.0
    print(
        "DWE GAME AUDIT "
        f"game={gid} actions={expected} logged={logged} coverage={game_cov:.3f}",
        flush=True,
    )

# GhostBridgePreMove is required before every real environment move. A brief may be
# generated for a move that never executes (for example an analyzer retry), so
# the invariant is executed_move_keys ⊆ prepared/logged GhostBridgePreMove move keys.
ghostbridge_premove_records = []
if GHOSTBRIDGE_PRE_MOVE_LOG.exists():
    for raw in GHOSTBRIDGE_PRE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        try:
            ghostbridge_premove_records.append(json.loads(raw))
        except json.JSONDecodeError:
            print("GHOSTBRIDGE_PREMOVE AUDIT malformed line:", raw[:240], flush=True)

ghostbridge_premove_keys = {
    (_dwe_game_key(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in ghostbridge_premove_records
    if item.get("move") is not None
}
executed_keys = {
    (_dwe_game_key(item.get("game_id", "unknown")), int(item.get("move", -1)))
    for item in records
    if item.get("move") is not None
}
missing_ghostbridge_premove = sorted(executed_keys - ghostbridge_premove_keys)
print(
    "GHOSTBRIDGE_PREMOVE PRE-MOVE AUDIT "
    f"executed_moves={len(executed_keys)} briefs={len(ghostbridge_premove_keys)} "
    f"missing={len(missing_ghostbridge_premove)}",
    flush=True,
)
if missing_ghostbridge_premove:
    print("WARNING: GhostBridgePreMove coverage gap:", repr(missing_ghostbridge_premove[:20]), flush=True)
else:
    print("GHOSTBRIDGE_PREMOVE AUDIT PASS: every executed move had a pre-move brief", flush=True)

if coverage < 0.999999:
    message = (
        "DWE per-move exploit logging coverage is incomplete: "
        f"{logged_moves}/{recorded_actions} ({coverage:.3%})."
    )
    if DWE_STRICT_LOG_COVERAGE:
        raise RuntimeError(message)
    print("WARNING:", message, flush=True)
else:
    print("GHOSTBRIDGE ADL AUDIT PASS: every recorded action has post-move ADL", flush=True)

if stop_loss:
    print(f"WARNING: {stop_loss} STOP_LOSS decisions were logged", flush=True)
else:
    print("ACTION CAP AUDIT PASS: no DWE STOP_LOSS decisions; all games uncapped", flush=True)


## 12. Final 3.57-control invariant audit


In [ ]:
# === GHOSTBRIDGE v5 COMPLETENESS / ZERO-DEBT SUMMARY ===
for _gbv5_game_id in sorted(_GBV5_RUNTIME.steps):
    _GBV5_RUNTIME.assert_zero_debt(_gbv5_game_id)
_gbv5_summary = GhostBridgeV5Introspection.snapshot(_GBV5_RUNTIME)
(WORKING_DIR / "ghostbridge_v5_summary.json").write_text(
    json.dumps(_gbv5_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("GHOSTBRIDGE v5 ZERO-DEBT PASS", json.dumps(_gbv5_summary, sort_keys=True), flush=True)


In [ ]:
# === 3.57 CONTROL / GHOSTBRIDGE PRE-MOVE FINAL INVARIANT AUDIT ===
from collections import Counter

_gb_records = []
if GHOSTBRIDGE_PRE_MOVE_LOG.exists():
    for _raw in GHOSTBRIDGE_PRE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        try:
            _gb_records.append(json.loads(_raw))
        except Exception:
            pass

_adl_records = []
if DWE_MOVE_LOG.exists():
    for _raw in DWE_MOVE_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        try:
            _adl_records.append(json.loads(_raw))
        except Exception:
            pass

_committed_actions = sum(_run_actions(run) for run in (getattr(bm, "game_runs", []) or []))
_gb_keys = {(str(x.get("game_id")), int(x.get("move", -1))) for x in _gb_records}
_adl_keys = {(str(x.get("game_id")), int(x.get("move", -1))) for x in _adl_records}

# Normalize by game key because analyzer and GameAPI objects can expose full ids differently.
def _norm_pair(pair):
    gid, move = pair
    try:
        return (_game_key(gid), int(move))
    except Exception:
        return (str(gid), int(move))

_gb_norm = {_norm_pair(x) for x in _gb_keys}
_adl_norm = {_norm_pair(x) for x in _adl_keys}
_missing_pre = sorted(_adl_norm - _gb_norm)

if _missing_pre:
    print(f"WARNING: committed moves without GhostBridge PRE-MOVE plan: {_missing_pre[:20]}", flush=True)
if len(_adl_norm) < _committed_actions:
    print(
        f"WARNING: solver-history/action-hook metric mismatch: "
        f"hook_logged={len(_adl_norm)} solver_history={_committed_actions}",
        flush=True,
    )

print(
    "TRUE SCORED RUN COMPLETE "
    f"baseline=wellkilo/arc3lab-duck-v12-control-seed-20260819 "
    f"recorded_control_score=3.57 seed={CONTROL_SEED} model={ANALYZER_MODEL_ID} "
    f"games={len(getattr(bm, 'game_runs', []) or [])} "
    f"actions={_committed_actions} ghostbridge_pre_move={len(_gb_norm)} "
    f"post_move_adl={len(_adl_norm)} adl_debt={max(0, _committed_actions-len(_adl_norm))} "
    f"submission={SUBMISSION_PATH}",
    flush=True,
)


# AUTOLOAD_FINAL: both startup audits must still exist at the end.
for _path in (WORKING_DIR / "auto_input_manifest.json", WORKING_DIR / "auto_runtime_audit.json"):
    if not _path.is_file():
        raise RuntimeError(f"AUTOLOAD FINAL AUDIT MISSING: {_path}")
print("AUTOLOAD FINAL AUDIT PASS: inputs + runtime + model were validated before gameplay", flush=True)
